### Setup

In [ ]:
%load_ext autoreload
%autoreload 2
# %reload_ext autoreload

In [ ]:
import sys
sys.path.insert(0, '../functions')

In [ ]:
import torch
import torch.nn as nn
from captum.attr import LRP
from captum.attr._utils.lrp_rules import EpsilonRule, IdentityRule

import numpy as np
import pandas as pd
import random
import json
from pathlib import Path

from scipy.stats import gaussian_kde

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from models import DANN
from data_loading import prepare_dataloaders, load_subject_data_flexible
from metrics import evaluate
from training_utils import get_optimizer, EarlyStopping, BestModelCheckpoint, initialise_log, train_epoch, get_alpha
from pipeline import train_dann_single_fold, run_loso

torch.manual_seed(25)
np.random.seed(25)
random.seed(25)

In [ ]:
from explainability import (
    DANNClassifierWrapper, load_model_for_xai,
    fold_batchnorm, verify_folding,
    lrp_epsilon_conv2d, lrp_zplus_conv2d,
    lrp_linear, lrp_avgpool,
    compute_lrp, run_lrp_full,
    load_lrp_attributions,
    compute_per_class_accuracy_106
)

from xai_utils import (
    assign_score_groups, get_grouped_order,
    get_score_grouped_order, classify_failure_groups,
    ch16_list
)

from xai_plots import plot_group_mean_attribution

In [ ]:
base_path = Path("..")

processed_path = base_path / "Data/processed"
results_path = base_path / "results"
multiclass_path = base_path / "Data/processed_multiclass"
ch16_path = base_path / "Data/processed_16ch"

lrp_attr_path = results_path / "lrp_attributions"
binary_attr_path = lrp_attr_path / "binary_cdan_16ch"
multiclass_attr_path = lrp_attr_path / "multiclass_16ch"
per_class_path = results_path / "per_class_accuracy"

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

### Config

In [ ]:
# ch16_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
#              'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

#### Valid Subjects

In [ ]:
# 106 subjects after removing S088, S092, S100 (incompatible sampling rate)
excluded = ['S088', 'S092', 'S100']

valid_subjects = []
for i in range(1, 110):
    subject_id = f'S{str(i).zfill(3)}'
    subject_file = processed_path / f'{subject_id}_processed.npz'
    if subject_file.exists() and subject_id not in excluded:
        valid_subjects.append(subject_id)

print(f"Valid subjects: {len(valid_subjects)}")
print(f"First five: {valid_subjects[:5]}")
print(f"Last five: {valid_subjects[-5:]}")

### End of config

### Checkpoint verification — model builder + load/compare against JSON

**Checkpoint verification (XAI prerequisite)** — confirms saved `.pt` weights load correctly and match JSON metrics, before any LRP work below.

In [ ]:
def build_model_for_run(n_classes=2, use_conditional_discriminator=True):
    return DANN(
        n_channels=16, n_timepoints=480, n_classes=n_classes,
        embed_dim=128, dropout_rate=0.25,
        use_conditional_discriminator=use_conditional_discriminator,
        feature_extractor_type='eegnet'
    )

def verify_checkpoint(subject, target_eval_loader, criterion, results, checkpoint_tag, n_classes=2):
    ckpt_path = results_path / "checkpoints" / checkpoint_tag / f"{subject}_{criterion}.pt"
    state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    model = build_model_for_run(n_classes=n_classes)
    model.load_state_dict(state_dict)
    model.eval()

    accuracy, kappa, auc, _, _ = evaluate(model, target_eval_loader)
    expected = results[subject][f"{criterion}_stopping"]

    return {
        "loaded_kappa": kappa,
        "expected_kappa": expected["final_kappa"],
        "kappa_match": abs(kappa - expected["final_kappa"]) < 1e-6,
    }

#### Run verification (full 106-subject check, both criteria)

In [ ]:
# with open(results_path / "loso_results_multiclass_16ch_with_checkpoints.json") as f:
#     multiclass_results = json.load(f)

# mismatches = []
# for subj in valid_subjects:
#     target_loader_data = prepare_dataloaders(
#         test_subject_id=subj, valid_subjects=valid_subjects,
#         use_16ch=True, use_multiclass=True
#     )
#     target_eval_loader = target_loader_data[2]
#     for crit in ["kappa", "composite"]:
#         r = verify_checkpoint(subj, target_eval_loader, crit, multiclass_results,
#                                checkpoint_tag="multiclass_16ch", n_classes=4)
#         if not r["kappa_match"]:
#             mismatches.append((subj, crit, r))

# print(f"{len(mismatches)} mismatches out of {len(valid_subjects)*2} checks")
# if mismatches:
#     print(mismatches[:10])

##### cdan

In [ ]:
# with open(results_path / "loso_results_cdan_16ch_RETRAIN.json") as f:
#     cdan_results = json.load(f)

# mismatches = []
# for subj in valid_subjects:
#     target_loader_data = prepare_dataloaders(
#         test_subject_id=subj, valid_subjects=valid_subjects,
#         use_16ch=True, use_multiclass=False
#     )
#     target_eval_loader = target_loader_data[2]
#     for crit in ["kappa", "composite"]:
#         r = verify_checkpoint(subj, target_eval_loader, crit, cdan_results,
#                                checkpoint_tag="cdan_16ch_binary", n_classes=2)
#         if not r["kappa_match"]:
#             mismatches.append((subj, crit, r))

# print(f"{len(mismatches)} mismatches out of {len(valid_subjects)*2} checks")
# if mismatches:
#     print(mismatches[:10])

### CLass DANN CLassifier wrapper

In [ ]:
class DANNClassifierWrapper(nn.Module):
    """Exposes only feature_extractor + classifier from DANN.
    Bypasses domain discriminator entirely, giving captum a clean
    single-output graph to trace attribution through."""
    def __init__(self, dann_model):
        super().__init__()
        self.feature_extractor = dann_model.feature_extractor
        self.classifier = dann_model.classifier

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.classifier(features)


def load_model_for_xai(subject, checkpoint_tag, n_classes=2, stopping_criterion='kappa'):
    ckpt_path = results_path / "checkpoints" / checkpoint_tag / f"{subject}_{stopping_criterion}.pt"
    state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model = DANN(
        n_channels=16, n_timepoints=480, n_classes=n_classes,
        embed_dim=128, dropout_rate=0.25,
        use_conditional_discriminator=True,
        feature_extractor_type='eegnet'
    )
    model.load_state_dict(state_dict)
    model.eval()
    return model

def set_lrp_rules(model):
    """
    Assigns LRP propagation rules to every leaf module.
    EpsilonRule for conv and linear (weight-bearing layers).
    IdentityRule for everything else (activations, norm, pooling, dropout)
    which passes relevance through unchanged.
    """
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            module.rule = EpsilonRule()
        elif isinstance(module, (nn.ELU, nn.ReLU,
                                  nn.BatchNorm2d, nn.BatchNorm1d,
                                  nn.AvgPool2d, nn.Dropout)):
            module.rule = IdentityRule()

def compute_lrp(subject, checkpoint_tag, n_classes=2,
                use_multiclass=False, stopping_criterion='kappa'):
    """
    Computes LRP attributions for one subject.
    Averages over correctly classified trials only, per true class.
    Returns:
        attributions: (n_trials, n_channels, n_timepoints), correctly classified only
        labels: (n_trials,) true labels for those trials
        mean_attr_per_class: dict {class_idx: mean attribution map (n_channels, n_timepoints)}
    """
    # load model and wrap
    model = load_model_for_xai(subject, checkpoint_tag,
                                n_classes=n_classes,
                                stopping_criterion=stopping_criterion)
    wrapper = DANNClassifierWrapper(model)
    wrapper.eval()

    # load data, shape: (n_trials, n_channels, n_timepoints)
    X, y = load_subject_data_flexible(
        subject, use_multiclass=use_multiclass, use_16ch=True
    )
    # EEGNet expects (batch, 1, n_channels, n_timepoints)
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.LongTensor(y)

    set_lrp_rules(wrapper)
    lrp = LRP(wrapper)

    all_attrs = []
    all_labels = []
    all_preds = []

    for i in range(len(X_tensor)):
        inp = X_tensor[i:i+1].clone().requires_grad_(True)

        with torch.no_grad():
            pred = wrapper(inp).argmax(dim=1).item()

        true_label = y_tensor[i].item()

        # only keep correctly classified trials
        if pred != true_label:
            continue

        attr = lrp.attribute(inp, target=true_label)
        # squeeze out batch and channel dims: (n_channels, n_timepoints)
        all_attrs.append(attr.detach().squeeze().numpy())
        all_labels.append(true_label)
        all_preds.append(pred)

    attributions = np.array(all_attrs)
    labels = np.array(all_labels)

    # mean attribution per class
    mean_attr_per_class = {}
    for cls in range(n_classes):
        mask = labels == cls
        if mask.sum() > 0:
            mean_attr_per_class[cls] = attributions[mask].mean(axis=0)
        else:
            mean_attr_per_class[cls] = None

    return attributions, labels, mean_attr_per_class

### folding function for batchnorm

In [ ]:
import copy

def fold_batchnorm(wrapper):
    """
    Returns a deep copy of wrapper with all Conv2d + BatchNorm2d pairs folded.
    Original wrapper is untouched.
    BatchNorm parameters absorbed into preceding Conv2d weights and bias.
    Predictions mathematically identical before and after folding.
    """
    folded = copy.deepcopy(wrapper)
    folded.eval()

    # exact paths: (block, conv_idx, bn_idx)
    pairs = [
        (folded.feature_extractor.temporal_conv,  0, 1),
        (folded.feature_extractor.depthwise_conv, 0, 1),
        (folded.feature_extractor.separable_conv, 1, 2),
    ]

    for block, conv_idx, bn_idx in pairs:
        conv = block[conv_idx]
        bn   = block[bn_idx]

        # scale: gamma / sqrt(running_var + eps)
        scale = bn.weight / torch.sqrt(bn.running_var + bn.eps)
        # bias:  beta - gamma * running_mean / sqrt(running_var + eps)
        bias  = bn.bias - bn.weight * bn.running_mean / torch.sqrt(bn.running_var + bn.eps)

        # absorb scale into conv weights
        # conv.weight shape: (out_channels, in_channels/groups, kH, kW)
        conv.weight.data = conv.weight.data * scale.view(-1, 1, 1, 1)

        # add bias term (conv.bias was None since bias=False at init)
        conv.bias = nn.Parameter(bias.clone())

        # replace BatchNorm with Identity
        block[bn_idx] = nn.Identity()

    return folded


def verify_folding(wrapper, folded, test_input):
    """
    Two checks:
    1. Max logit difference between original and folded < 1e-5
    2. No BatchNorm2d layers remaining in folded model
    """
    wrapper.eval()
    folded.eval()

    with torch.no_grad():
        orig_out = wrapper(test_input)
        fold_out = folded(test_input)

    max_diff = (orig_out - fold_out).abs().max().item()
    print(f"Max logit difference: {max_diff:.10f}")
    print(f"Folding correct: {max_diff < 1e-5}")

    bn_remaining = [
        name for name, m in folded.named_modules()
        if isinstance(m, nn.BatchNorm2d)
    ]
    print(f"BatchNorm2d layers remaining: {len(bn_remaining)}")
    if bn_remaining:
        print(f"  {bn_remaining}")

### test

#### class dann classifier wrapper test

In [ ]:
# attrs, labels, mean_by_class = compute_lrp(
#     subject='S007',
#     checkpoint_tag='cdan_16ch_binary',
#     n_classes=2,
#     use_multiclass=False,
#     stopping_criterion='kappa'
# )

# print(f"Correctly classified trials: {len(attrs)}")
# print(f"Attribution shape per trial: {attrs[0].shape}")
# print(f"Classes present: {np.unique(labels)}")
# for cls, mean_attr in mean_by_class.items():
#     if mean_attr is not None:
#         print(f"Class {cls} mean attribution shape: {mean_attr.shape}, "
#               f"range: [{mean_attr.min():.4f}, {mean_attr.max():.4f}]")

#### Folding function test

In [ ]:
model = load_model_for_xai('S007', 'cdan_16ch_binary', n_classes=2)
wrapper = DANNClassifierWrapper(model)
wrapper.eval()

folded = fold_batchnorm(wrapper)

# use one real trial as test input
X, y = load_subject_data_flexible('S007', use_multiclass=False, use_16ch=True)
test_input = torch.FloatTensor(X[:1])

verify_folding(wrapper, folded, test_input)

### LRP

#### Subset List for LRP - Multiclass

##### analysis on the performers

In [ ]:
# kappas = {
#     subj: multiclass_results[subj]["kappa_stopping"]["final_kappa"]
#     for subj in valid_subjects
# }

# sorted_subjects = sorted(kappas.items(), key=lambda x: x[1], reverse=True)
# mean_kappa = sum(kappas.values()) / len(kappas)

# top5 = sorted_subjects[:5]
# bottom5 = sorted_subjects[-5:]

# typical = sorted(kappas.items(), key=lambda x: abs(x[1] - mean_kappa))[:5]

# print("Top 5 (best performers):", top5)
# print("Bottom 5 (negative performers, by raw kappa):", bottom5)
# print("Mean kappa:", mean_kappa)
# print("Closest to mean (typical):", typical)

# # check overlap against already-documented failure-mode subjects
# group_a = {"S006", "S018", "S037"}
# group_b = {"S009", "S020", "S039", "S067", "S097", "S107"}
# bottom5_ids = {s for s, k in bottom5}

# print("\nBottom 5 that are also Group A (selective collapse):", bottom5_ids & group_a)
# print("Bottom 5 that are also Group B (flat near-chance):", bottom5_ids & group_b)
# print("Group A subjects NOT in bottom 5 by raw kappa:", group_a - bottom5_ids)

#### lrp initiation

In [ ]:
xai_subjects = {
    "top": ["S007", "S085"],
    "bottom_kappa": ["S018", "S097", "S107"],
    "group_a_collapse": ["S037", "S006"],
    "group_b_flat": ["S039", "S020"],
    "typical": ["S080", "S065", "S030", "S108"],
}

xai_subset = [s for group in xai_subjects.values() for s in group]
print(f"Total subjects: {len(xai_subset)}")
print(xai_subset)

##### Checks for lrp

In [ ]:
X, y = load_subject_data_flexible('S007', use_multiclass=False, use_16ch=True)
print(X.shape)

In [ ]:
# load S007 model, wrap it, print the full named module tree
model = load_model_for_xai('S007', 'cdan_16ch_binary', n_classes=2)
wrapper = DANNClassifierWrapper(model)

for name, module in wrapper.named_modules():
    print(name, '|', type(module).__name__)

In [ ]:
model = load_model_for_xai('S007', 'cdan_16ch_binary', n_classes=2)
wrapper = DANNClassifierWrapper(model)
folded = fold_batchnorm(wrapper)
folded.eval()

X, y = load_subject_data_flexible('S007', use_multiclass=False, use_16ch=True)
test_input = torch.FloatTensor(X[:1])

# trace shapes through each block
with torch.no_grad():
    after_temporal = folded.feature_extractor.temporal_conv(test_input)
    print("after temporal_conv:", after_temporal.shape)

    after_depthwise = folded.feature_extractor.depthwise_conv(after_temporal)
    print("after depthwise_conv:", after_depthwise.shape)

    after_separable = folded.feature_extractor.separable_conv(after_depthwise)
    print("after separable_conv:", after_separable.shape)

    flattened = after_separable.view(after_separable.size(0), -1)
    print("after flatten:", flattened.shape)

    projected = folded.feature_extractor.projection(flattened)
    print("after projection:", projected.shape)

#### LRP helper functions and compute_lrp

In [ ]:
def lrp_epsilon_conv2d(layer, activation, relevance, epsilon=1e-6):
    """
    Epsilon rule for Conv2d.
    Propagates relevance backward through a conv layer using saved activation.
    activation: input to this layer, shape as it entered the layer
    relevance: relevance arriving at this layer's output
    """
    activation = activation.detach().requires_grad_(True)
    z = layer(activation)
    # stabilise denominator with epsilon
    s = relevance / (z + epsilon * z.sign().clamp(min=1e-6) + 1e-6)
    z.backward(s)
    return activation.grad * activation

In [ ]:
def lrp_zplus_conv2d(layer, activation, relevance, epsilon=1e-6):
    """
    Z-plus rule for Conv2d.
    Only positive contributions counted, cleaner input-level attributions.
    Used optionally for first Conv2d layer only (temporal_conv.0).
    """
    # temporarily zero negative weights
    weight_orig = layer.weight.data.clone()
    layer.weight.data = weight_orig.clamp(min=0)

    activation = activation.detach().requires_grad_(True)
    z = layer(activation)
    s = relevance / (z + epsilon)
    z.backward(s)
    grad = activation.grad * activation

    # restore original weights
    layer.weight.data = weight_orig
    return grad

In [ ]:
def lrp_linear(layer, activation, relevance, epsilon=1e-6):
    """
    Epsilon rule for Linear layers.
    """
    activation = activation.detach().requires_grad_(True)
    z = layer(activation)
    s = relevance / (z + epsilon * z.sign().clamp(min=1e-6) + 1e-6)
    z.backward(s)
    return activation.grad * activation

In [ ]:
def lrp_avgpool(layer, activation, relevance):
    """
    Pass-through for AvgPool2d.
    Distributes relevance uniformly across the pooling window,
    inverse of what average pooling did in the forward pass.
    """
    activation = activation.detach().requires_grad_(True)
    z = layer(activation)
    s = relevance / (z + 1e-6)
    z.backward(s)
    return activation.grad * activation


In [ ]:
def compute_lrp(
    subject,
    checkpoint_tag,
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    first_layer_rule='epsilon',
    epsilon=1e-6,
    trial_type='correct',
):
    """
    Manual LRP implementation for DANNClassifierWrapper + EEGNet.
    trial_type:
        'correct'      — only correctly classified trials, attribution
                         targets true label (original behaviour)
        'misclassified'— only misclassified trials, attribution targets
                         predicted label (what model actually decided)
        'all'          — both sets returned separately as a tuple
    Returns:
        if trial_type != 'all':
            attributions: (n_trials, n_channels, n_timepoints)
            labels: (n_trials,) true labels
            mean_attr_per_class: dict {class_idx: mean attr (n_channels, n_timepoints)}
            [if misclassified: also stores predicted labels separately]
        if trial_type == 'all':
            (correct_output, misclassified_output) where each is the above tuple
    """
    model = load_model_for_xai(
        subject, checkpoint_tag,
        n_classes=n_classes,
        stopping_criterion=stopping_criterion
    )
    wrapper = DANNClassifierWrapper(model)
    folded = fold_batchnorm(wrapper)
    folded.eval()

    fe = folded.feature_extractor
    clf = folded.classifier

    X, y = load_subject_data_flexible(
        subject, use_multiclass=use_multiclass, use_16ch=True
    )
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.LongTensor(y)

    correct_attrs, correct_labels = [], []
    misc_attrs, misc_true_labels, misc_pred_labels = [], [], []

    for i in range(len(X_tensor)):
        inp = X_tensor[i:i+1]

        with torch.no_grad():
            a0 = inp
            a1 = fe.temporal_conv(a0)
            a2 = fe.depthwise_conv(a1)
            a3 = fe.separable_conv(a2)
            a4 = a3.view(a3.size(0), -1)
            a5 = fe.projection(a4)
            a6 = clf.network[0](a5)
            a7 = torch.relu(a6)
            a8 = a7
            a9 = clf.network[3](a8)

            pred = a9.argmax(dim=1).item()
            true_label = y_tensor[i].item()

        is_correct = (pred == true_label)

        # decide whether to compute attribution for this trial
        if trial_type == 'correct' and not is_correct:
            continue
        if trial_type == 'misclassified' and is_correct:
            continue

        # attribution targets predicted class for misclassified,
        # true class for correct
        attr_target = true_label if is_correct else pred

        # backward relevance pass
        R = torch.zeros_like(a9)
        R[0, attr_target] = a9[0, attr_target]

        R = lrp_linear(clf.network[3], a8, R, epsilon)
        R = lrp_linear(clf.network[0], a5, R, epsilon)
        R = lrp_linear(fe.projection, a4, R, epsilon)
        R = R.view(a3.shape)

        with torch.no_grad():
            pre_pool_sep = fe.separable_conv[:4](a2)
        R = lrp_avgpool(fe.separable_conv[4], pre_pool_sep, R)
        R = lrp_epsilon_conv2d(fe.separable_conv[1], pre_pointwise := fe.separable_conv[0](a2), R, epsilon)
        R = lrp_epsilon_conv2d(fe.separable_conv[0], a2, R, epsilon)

        with torch.no_grad():
            pre_pool_dw = fe.depthwise_conv[:3](a1)
        R = lrp_avgpool(fe.depthwise_conv[3], pre_pool_dw, R)
        R = lrp_epsilon_conv2d(fe.depthwise_conv[0], a1, R, epsilon)

        if first_layer_rule == 'zplus':
            R = lrp_zplus_conv2d(fe.temporal_conv[0], a0, R, epsilon)
        else:
            R = lrp_epsilon_conv2d(fe.temporal_conv[0], a0, R, epsilon)

        attr = R.detach().squeeze().numpy()

        if is_correct:
            correct_attrs.append(attr)
            correct_labels.append(true_label)
        else:
            misc_attrs.append(attr)
            misc_true_labels.append(true_label)
            misc_pred_labels.append(pred)

    def build_output(attrs, labels, n_classes):
        attributions = np.array(attrs) if attrs else np.array([])
        labels_arr = np.array(labels)
        mean_attr_per_class = {}
        for cls in range(n_classes):
            mask = labels_arr == cls
            if mask.sum() > 0:
                mean_attr_per_class[cls] = attributions[mask].mean(axis=0)
            else:
                mean_attr_per_class[cls] = None
        return attributions, labels_arr, mean_attr_per_class

    correct_out = build_output(correct_attrs, correct_labels, n_classes)

    misc_out = (
        np.array(misc_attrs) if misc_attrs else np.array([]),
        np.array(misc_true_labels),
        np.array(misc_pred_labels),
        build_output(misc_attrs, misc_pred_labels, n_classes)[2]
    )

    if trial_type == 'correct':
        return correct_out
    elif trial_type == 'misclassified':
        return misc_out
    else:
        return correct_out, misc_out

### Grouping functions

In [ ]:
def assign_score_groups(kappas_dict, std_multiplier=0.7, chance_threshold=0.1):
    """
    Assigns score-based groups to subjects using std deviation bands.
    Groups derived from full population distribution, not hardcoded.

    kappas_dict: {subject_id: kappa_value} for the full population
    std_multiplier: controls band width, 0.7 chosen by minimising
                    within-group variance while maintaining full
                    5-group coverage on XAI subset
    chance_threshold: kappa below this = negative transfer (default 0.1,
                      matches existing above-chance threshold in thesis)

    Returns:
        group_assignments: {subject_id: group_name}
        boundaries: dict with mean, std, low and high boundaries
    """
    kappa_values = np.array(list(kappas_dict.values()))
    mean_k = kappa_values.mean()
    std_k = kappa_values.std()

    low_boundary = mean_k - (std_multiplier * std_k)
    high_boundary = mean_k + (std_multiplier * std_k)

    group_assignments = {}
    for subj, k in kappas_dict.items():
        if k < chance_threshold:
            group_assignments[subj] = 'negative_transfer'
        elif k < low_boundary:
            group_assignments[subj] = 'low'
        elif k < mean_k:
            group_assignments[subj] = 'mid_low'
        elif k < high_boundary:
            group_assignments[subj] = 'mid_high'
        else:
            group_assignments[subj] = 'high'

    boundaries = {
        'mean': round(mean_k, 4),
        'std': round(std_k, 4),
        'std_multiplier': std_multiplier,
        'low_boundary': round(low_boundary, 4),
        'high_boundary': round(high_boundary, 4),
        'chance_threshold': chance_threshold,
    }

    return group_assignments, boundaries

In [ ]:
def get_score_grouped_order(score_groups, lrp_results, xai_subset):
    """
    Returns subjects ordered by score group, top to bottom:
    high -> mid_high -> mid_low -> low -> negative_transfer.
    Only includes subjects present in both lrp_results and xai_subset.
    """
    group_order = ['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']
    ordered = []
    subject_to_group = {}

    for grp in group_order:
        for subj in xai_subset:
            if subj in lrp_results and score_groups.get(subj) == grp:
                ordered.append(subj)
                subject_to_group[subj] = grp

    return ordered, subject_to_group


def plot_score_grouped_bars(lrp_results, score_groups, xai_subset,
                            boundaries, kappas_dict,
                            class_names=None, save=True):
    """
    Channel importance bar chart, all XAI subjects, ordered by score group.
    Same 4-column structure as category-grouped version:
    left-shared | left-per-subject | right-shared | right-per-subject
    """
    if class_names is None:
        class_names = ['left', 'right']

    grouped_order, subject_to_group = get_score_grouped_order(
        score_groups, lrp_results, xai_subset
    )
    n_subjects = len(grouped_order)

    global_xmax = max(
        np.abs(lrp_results[s][2][cls]).mean(axis=1).max()
        for s in grouped_order
        for cls in [0, 1]
        if lrp_results[s][2][cls] is not None
    )

    col_specs = [(0, 'shared'), (0, 'per'), (1, 'shared'), (1, 'per')]
    colors = ['#d62728' if ch in ['C3', 'Cz', 'C4'] else '#1f77b4'
              for ch in ch16_list]

    fig, axes = plt.subplots(n_subjects, 4, figsize=(20, 2.5 * n_subjects))

    for row_idx, subj in enumerate(grouped_order):
        _, _, mean_by_class = lrp_results[subj]
        group = subject_to_group[subj]
        kappa = kappas_dict[subj]

        for col_idx, (cls, scale) in enumerate(col_specs):
            ax = axes[row_idx, col_idx]
            mean_attr = mean_by_class[cls]
            if mean_attr is None:
                ax.axis('off')
                continue

            mean_abs = np.abs(mean_attr).mean(axis=1)
            xmax = global_xmax if scale == 'shared' else mean_abs.max() * 1.1

            ax.barh(ch16_list, mean_abs, color=colors)
            ax.set_xlim(0, xmax)
            ax.invert_yaxis()
            ax.set_xlabel('mean |relevance|', fontsize=7)

            if col_idx in [1, 3]:
                ax.set_yticklabels([])
            else:
                ax.tick_params(axis='y', labelsize=7)

            if col_idx == 0:
                ax.set_title(
                    f'{subj} [{group} | κ={kappa:.3f}] | '
                    f'{class_names[cls]} | shared',
                    fontsize=8, loc='left'
                )
            else:
                ax.set_title(
                    f'{class_names[cls]} | '
                    f'{"shared" if scale == "shared" else "per-subject"}',
                    fontsize=8
                )

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d62728', label='motor cortex (C3/Cz/C4)'),
        Patch(facecolor='#1f77b4', label='other channels')
    ]
    axes[0, 3].legend(handles=legend_elements, loc='lower right', fontsize=7)

    plt.suptitle(
        f'LRP channel importance — score groups '
        f'(boundaries: low<{boundaries["low_boundary"]}, '
        f'mid_low<{boundaries["mean"]}, '
        f'mid_high<{boundaries["high_boundary"]})',
        fontsize=11, y=1.001
    )
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / 'lrp_score_grouped_bars_binary.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def plot_score_grouped_heatmaps(lrp_results, score_groups, xai_subset,
                                boundaries, kappas_dict, class_names=None,
                                sfreq=160, save=True):
    """
    Heatmaps, all XAI subjects, ordered by score group.
    Same 4-column structure as category-grouped version.
    """

    if class_names is None:
        class_names = ['left', 'right']

    grouped_order, subject_to_group = get_score_grouped_order(
        score_groups, lrp_results, xai_subset
    )
    n_subjects = len(grouped_order)

    global_vmax = max(
        np.abs(lrp_results[s][2][cls]).max()
        for s in grouped_order
        for cls in [0, 1]
        if lrp_results[s][2][cls] is not None
    )

    timepoints = 480
    tick_positions = list(range(0, timepoints, 80))
    tick_labels = [f"{t/sfreq:.1f}" for t in tick_positions]
    col_specs = [(0, 'shared'), (0, 'per'), (1, 'shared'), (1, 'per')]

    fig, axes = plt.subplots(
        n_subjects, 4,
        figsize=(24, 3.5 * n_subjects)
    )

    for row_idx, subj in enumerate(grouped_order):
        _, _, mean_by_class = lrp_results[subj]
        group = subject_to_group[subj]
        kappa = kappas_dict[subj]

        subj_vmax = max(
            np.abs(mean_by_class[cls]).max()
            for cls in [0, 1]
            if mean_by_class[cls] is not None
        )

        for col_idx, (cls, scale) in enumerate(col_specs):
            ax = axes[row_idx, col_idx]
            mean_attr = mean_by_class[cls]
            if mean_attr is None:
                ax.axis('off')
                continue

            vmax = global_vmax if scale == 'shared' else subj_vmax
            show_yticks = col_idx in [0, 2]

            sns.heatmap(
                mean_attr, ax=ax,
                cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                xticklabels=False,
                yticklabels=ch16_list if show_yticks else False,
                cbar=True,
                cbar_kws={'label': 'relevance', 'shrink': 0.8}
            )
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=0, fontsize=7)
            ax.set_xlabel('time (s)', fontsize=8)

            if show_yticks:
                ax.tick_params(axis='y', labelsize=7)

            if col_idx == 0:
                ax.set_title(
                    f'{subj} [{group} | κ={kappa:.3f}] | '
                    f'{class_names[cls]} | shared',
                    fontsize=8, loc='left'
                )
            else:
                ax.set_title(
                    f'{class_names[cls]} | '
                    f'{"shared" if scale == "shared" else "per-subject"}',
                    fontsize=8
                )

    plt.suptitle(
        f'LRP heatmaps — score groups '
        f'(boundaries: low<{boundaries["low_boundary"]}, '
        f'mid_low<{boundaries["mean"]}, '
        f'mid_high<{boundaries["high_boundary"]})',
        fontsize=11, y=1.001
    )
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / 'lrp_score_grouped_heatmaps_binary.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()

#### check epsilon vs zplus

In [ ]:
# epsilon throughout
attrs_eps, labels_eps, mean_eps = compute_lrp(
    subject='S007',
    checkpoint_tag='cdan_16ch_binary',
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    first_layer_rule='epsilon',
)

print(f"Correctly classified trials: {len(attrs_eps)}")
print(f"Attribution shape: {attrs_eps[0].shape}")
for cls, m in mean_eps.items():
    if m is not None:
        print(f"Class {cls} | range [{m.min():.4f}, {m.max():.4f}] | "
              f"mean abs {np.abs(m).mean():.4f}")

print()

# zplus on first layer
attrs_zp, labels_zp, mean_zp = compute_lrp(
    subject='S007',
    checkpoint_tag='cdan_16ch_binary',
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    first_layer_rule='zplus',
)

print(f"Correctly classified trials (zplus): {len(attrs_zp)}")
for cls, m in mean_zp.items():
    if m is not None:
        print(f"Class {cls} | range [{m.min():.4f}, {m.max():.4f}] | "
              f"mean abs {np.abs(m).mean():.4f}")

#### Plots

In [ ]:


def plot_lrp_heatmap(mean_attr_per_class, subject, rule, n_classes=2,
                     class_names=None, sfreq=160):
    """
    Plots mean LRP attribution heatmap per class.
    X axis: time in seconds. Y axis: channels.
    Diverging colormap: blue=negative, white=zero, red=positive relevance.
    """
    if class_names is None:
        class_names = ['left', 'right'] if n_classes == 2 else \
                      ['left', 'right', 'both-fists', 'both-feet']

    n_plots = sum(1 for v in mean_attr_per_class.values() if v is not None)
    fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    # shared colormap scale across classes for fair comparison
    vmax = max(
        np.abs(v).max()
        for v in mean_attr_per_class.values() if v is not None
    )

    timepoints = mean_attr_per_class[0].shape[1] if mean_attr_per_class[0] is not None \
                 else mean_attr_per_class[1].shape[1]
    time_labels = [f"{t/sfreq:.1f}" for t in range(timepoints)]
    # show only every 80th label (every 0.5s) to avoid clutter
    tick_positions = list(range(0, timepoints, 80))
    tick_labels = [time_labels[t] for t in tick_positions]

    plot_idx = 0
    for cls, mean_attr in mean_attr_per_class.items():
        if mean_attr is None:
            continue
        ax = axes[plot_idx]
        sns.heatmap(
            mean_attr,
            ax=ax,
            cmap='RdBu_r',
            vmin=-vmax, vmax=vmax,
            xticklabels=False,
            yticklabels=ch16_list,
            cbar_kws={'label': 'relevance'}
        )
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=0, fontsize=9)
        ax.set_xlabel('time (s)', fontsize=10)
        ax.set_ylabel('channel', fontsize=10)
        ax.set_title(f'{subject} | class: {class_names[cls]} | rule: {rule}',
                     fontsize=11)
        plot_idx += 1

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_lrp_analysis(mean_attr_per_class, subject, n_classes=2,
                      class_names=None, sfreq=160, save=True):
    """
    Combined LRP analysis plot per subject.
    Top row: channel-by-time heatmap per class.
    Bottom row: mean absolute relevance per channel (collapsed across time).
    Epsilon rule assumed throughout.
    """
    if class_names is None:
        class_names = ['left', 'right'] if n_classes == 2 else \
                      ['left', 'right', 'both-fists', 'both-feet']

    valid_classes = [cls for cls, v in mean_attr_per_class.items() if v is not None]
    n_cols = len(valid_classes)

    fig, axes = plt.subplots(
        2, n_cols,
        figsize=(8 * n_cols, 9),
        gridspec_kw={'height_ratios': [3, 1]}
    )
    if n_cols == 1:
        axes = axes.reshape(-1, 1)

    # shared colormap scale across classes
    vmax = max(np.abs(mean_attr_per_class[cls]).max() for cls in valid_classes)

    timepoints = mean_attr_per_class[valid_classes[0]].shape[1]
    tick_positions = list(range(0, timepoints, 80))
    tick_labels = [f"{t/sfreq:.1f}" for t in tick_positions]

    for col_idx, cls in enumerate(valid_classes):
        mean_attr = mean_attr_per_class[cls]

        # top: heatmap
        ax_heat = axes[0, col_idx]
        sns.heatmap(
            mean_attr, ax=ax_heat,
            cmap='RdBu_r', vmin=-vmax, vmax=vmax,
            xticklabels=False, yticklabels=ch16_list,
            cbar_kws={'label': 'relevance'}
        )
        ax_heat.set_xticks(tick_positions)
        ax_heat.set_xticklabels(tick_labels, rotation=0, fontsize=9)
        ax_heat.set_xlabel('time (s)', fontsize=10)
        ax_heat.set_ylabel('channel', fontsize=10)
        ax_heat.set_title(
            f'{subject} | class: {class_names[cls]} | epsilon rule',
            fontsize=11
        )

        # bottom: mean absolute relevance per channel, collapsed across time
        ax_bar = axes[1, col_idx]
        mean_abs = np.abs(mean_attr).mean(axis=1)  # (16,)
        colors = ['#d62728' if ch in ['C3', 'C4', 'Cz'] else '#1f77b4'
                  for ch in ch16_list]
        ax_bar.barh(ch16_list, mean_abs, color=colors)
        ax_bar.set_xlabel('mean |relevance|', fontsize=10)
        ax_bar.set_ylabel('channel', fontsize=10)
        ax_bar.set_title(f'channel importance | class: {class_names[cls]}',
                         fontsize=10)
        ax_bar.invert_yaxis()

    # legend for bar chart colours
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d62728', label='motor cortex (C3/Cz/C4)'),
        Patch(facecolor='#1f77b4', label='other channels')
    ]
    axes[1, -1].legend(handles=legend_elements, loc='lower right', fontsize=8)

    plt.suptitle(f'LRP attribution analysis — {subject}', fontsize=13, y=1.01)
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / f'lrp_analysis_{subject}.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

rule_data = [('epsilon', mean_eps), ('zplus', mean_zp)]
class_names = ['left', 'right']
timepoints = 480
tick_positions = list(range(0, timepoints, 80))
tick_labels = [f"{t/160:.1f}" for t in tick_positions]

for row_idx, (rule, mean_attr_per_class) in enumerate(rule_data):
    vmax = max(np.abs(v).max() for v in mean_attr_per_class.values() if v is not None)
    for col_idx, cls in enumerate([0, 1]):
        ax = axes[row_idx, col_idx]
        mean_attr = mean_attr_per_class[cls]
        if mean_attr is None:
            ax.axis('off')
            continue
        sns.heatmap(
            mean_attr, ax=ax, cmap='RdBu_r',
            vmin=-vmax, vmax=vmax,
            xticklabels=False, yticklabels=ch16_list,
            cbar_kws={'label': 'relevance'}
        )
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=0, fontsize=9)
        ax.set_xlabel('time (s)', fontsize=10)
        ax.set_ylabel('channel', fontsize=10)
        ax.set_title(f'S007 | class: {class_names[cls]} | rule: {rule}', fontsize=11)

plt.tight_layout()
plt.savefig(results_path / 'figures' / 'lrp_S007_epsilon_vs_zplus.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# S007 - already computed, just plot
plot_lrp_analysis(mean_eps, subject='S007', n_classes=2)

# S018 - compute then plot
attrs_s018, labels_s018, mean_s018 = compute_lrp(
    subject='S018',
    checkpoint_tag='cdan_16ch_binary',
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    first_layer_rule='epsilon',
)

print(f"S018 correctly classified trials: {len(attrs_s018)}")
print(f"S018 class distribution: {dict(zip(*np.unique(labels_s018, return_counts=True)))}")

plot_lrp_analysis(mean_s018, subject='S018', n_classes=2)

In [ ]:
remaining = [s for s in xai_subset if s not in ['S007', 'S018']]
print(remaining)

In [ ]:
lrp_results = {}

# already computed
lrp_results['S007'] = (attrs_eps, labels_eps, mean_eps)
lrp_results['S018'] = (attrs_s018, labels_s018, mean_s018)

remaining = [s for s in xai_subset if s not in ['S007', 'S018']]

for subj in remaining:
    print(f"Computing {subj}...")
    attrs, labels, mean_by_class = compute_lrp(
        subject=subj,
        checkpoint_tag='cdan_16ch_binary',
        n_classes=2,
        use_multiclass=False,
        stopping_criterion='kappa',
        first_layer_rule='epsilon',
    )
    lrp_results[subj] = (attrs, labels, mean_by_class)
    print(f"  {len(attrs)} correctly classified trials")

print(f"\nDone: {list(lrp_results.keys())}")

In [ ]:
def get_grouped_order(xai_subjects, lrp_results):
    """Returns subjects in group order, with group label per subject."""
    ordered = []
    subject_to_group = {}
    for group_name, subjects in xai_subjects.items():
        for s in subjects:
            if s in lrp_results:
                ordered.append(s)
                subject_to_group[s] = group_name
    return ordered, subject_to_group


def plot_grouped_bars(lrp_results, xai_subjects, class_names=None,
                      title_tag='binary CDAN-16ch', save=True):
    """
    Figure 1: channel importance bars, all subjects.
    13 rows x 4 cols: left-shared | left-per-subject | right-shared | right-per-subject
    Subjects ordered top to bottom by group.
    """
    if class_names is None:
        class_names = ['left', 'right']

    grouped_order, subject_to_group = get_grouped_order(xai_subjects, lrp_results)
    n_subjects = len(grouped_order)

    # global x max for shared scale
    global_xmax = max(
        np.abs(lrp_results[s][2][cls]).mean(axis=1).max()
        for s in grouped_order
        for cls in [0, 1]
        if lrp_results[s][2][cls] is not None
    )

    col_specs = [(0, 'shared'), (0, 'per'), (1, 'shared'), (1, 'per')]
    colors = ['#d62728' if ch in ['C3', 'Cz', 'C4'] else '#1f77b4'
              for ch in ch16_list]

    fig, axes = plt.subplots(n_subjects, 4, figsize=(20, 2.5 * n_subjects))

    for row_idx, subj in enumerate(grouped_order):
        _, _, mean_by_class = lrp_results[subj]
        group = subject_to_group[subj]

        for col_idx, (cls, scale) in enumerate(col_specs):
            ax = axes[row_idx, col_idx]
            mean_attr = mean_by_class[cls]
            if mean_attr is None:
                ax.axis('off')
                continue

            mean_abs = np.abs(mean_attr).mean(axis=1)
            xmax = global_xmax if scale == 'shared' else mean_abs.max() * 1.1

            ax.barh(ch16_list, mean_abs, color=colors)
            ax.set_xlim(0, xmax)
            ax.invert_yaxis()
            ax.set_xlabel('mean |relevance|', fontsize=7)

            # show channel labels only on col 0 and col 2
            if col_idx in [1, 3]:
                ax.set_yticklabels([])
            else:
                ax.tick_params(axis='y', labelsize=7)

            # title row: include subject + group on col 0 only
            if col_idx == 0:
                ax.set_title(
                    f'{subj} [{group}] | {class_names[cls]} | shared',
                    fontsize=8, loc='left'
                )
            else:
                ax.set_title(
                    f'{class_names[cls]} | {"shared" if scale == "shared" else "per-subject"}',
                    fontsize=8
                )

    # legend on last subplot
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d62728', label='motor cortex (C3/Cz/C4)'),
        Patch(facecolor='#1f77b4', label='other channels')
    ]
    axes[0, 3].legend(handles=legend_elements, loc='lower right', fontsize=7)

    plt.suptitle(f'LRP channel importance — {title_tag}, all subjects',
        fontsize=12, y=1.001
    )
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / 'lrp_grouped_bars_binary.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def plot_grouped_heatmaps(lrp_results, xai_subjects, class_names=None,
                          title_tag='binary CDAN-16ch',sfreq=160, save=True):
    """
    Figure 2: heatmaps, all subjects.
    13 rows x 4 cols: left-shared | left-per-subject | right-shared | right-per-subject
    Subjects ordered top to bottom by group.
    """
    if class_names is None:
        class_names = ['left', 'right']

    grouped_order, subject_to_group = get_grouped_order(xai_subjects, lrp_results)
    n_subjects = len(grouped_order)

    # global vmax for shared colormap
    global_vmax = max(
        np.abs(lrp_results[s][2][cls]).max()
        for s in grouped_order
        for cls in [0, 1]
        if lrp_results[s][2][cls] is not None
    )

    timepoints = 480
    tick_positions = list(range(0, timepoints, 80))
    tick_labels = [f"{t/sfreq:.1f}" for t in tick_positions]

    col_specs = [(0, 'shared'), (0, 'per'), (1, 'shared'), (1, 'per')]

    fig, axes = plt.subplots(
        n_subjects, 4,
        figsize=(24, 3.5 * n_subjects)
    )

    for row_idx, subj in enumerate(grouped_order):
        _, _, mean_by_class = lrp_results[subj]
        group = subject_to_group[subj]

        subj_vmax = max(
            np.abs(mean_by_class[cls]).max()
            for cls in [0, 1]
            if mean_by_class[cls] is not None
        )

        for col_idx, (cls, scale) in enumerate(col_specs):
            ax = axes[row_idx, col_idx]
            mean_attr = mean_by_class[cls]
            if mean_attr is None:
                ax.axis('off')
                continue

            vmax = global_vmax if scale == 'shared' else subj_vmax
            show_yticks = col_idx in [0, 2]

            sns.heatmap(
                mean_attr, ax=ax,
                cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                xticklabels=False,
                yticklabels=ch16_list if show_yticks else False,
                cbar=True,
                cbar_kws={'label': 'relevance', 'shrink': 0.8}
            )
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=0, fontsize=7)
            ax.set_xlabel('time (s)', fontsize=8)

            if show_yticks:
                ax.tick_params(axis='y', labelsize=7)

            if col_idx == 0:
                ax.set_title(
                    f'{subj} [{group}] | {class_names[cls]} | shared',
                    fontsize=8, loc='left'
                )
            else:
                ax.set_title(
                    f'{class_names[cls]} | {"shared" if scale == "shared" else "per-subject"}',
                    fontsize=8
                )

    plt.suptitle(f'LRP heatmaps — {title_tag}, all subjects',
        fontsize=12, y=1.001
    )
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / 'lrp_grouped_heatmaps_binary.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()

#### plot_grouped_heatmaps_multiclass

In [ ]:
def plot_grouped_heatmaps_multiclass(lrp_results, xai_subjects,
                                      sfreq=160, save=True):
    """
    Heatmaps for 4-class multiclass LRP results.
    26 rows (2 per subject), 4 columns (shared/per-subject for 2 classes per row).
    Row 1 per subject: left (class 0) and right (class 1)
    Row 2 per subject: both-fists (class 2) and both-feet (class 3)
    """
    class_names = ['left', 'right', 'both-fists', 'both-feet']
    grouped_order, subject_to_group = get_grouped_order(xai_subjects, lrp_results)
    n_subjects = len(grouped_order)

    global_vmax = max(
        np.abs(lrp_results[s][2][cls]).max()
        for s in grouped_order
        for cls in range(4)
        if lrp_results[s][2][cls] is not None
    )

    timepoints = 480
    tick_positions = list(range(0, timepoints, 80))
    tick_labels = [f"{t/sfreq:.1f}" for t in tick_positions]

    # 2 rows per subject, 4 columns
    # row offset 0: classes 0 and 1 (left, right)
    # row offset 1: classes 2 and 3 (both-fists, both-feet)
    # columns: class_a shared | class_a per-subject | class_b shared | class_b per-subject
    fig, axes = plt.subplots(
        n_subjects * 2, 4,
        figsize=(24, 3.5 * n_subjects * 2)
    )

    for subj_idx, subj in enumerate(grouped_order):
        _, _, mean_by_class = lrp_results[subj]
        group = subject_to_group[subj]

        subj_vmax = max(
            np.abs(mean_by_class[cls]).max()
            for cls in range(4)
            if mean_by_class[cls] is not None
        )

        for row_offset, (cls_a, cls_b) in enumerate([(0, 1), (2, 3)]):
            row_idx = subj_idx * 2 + row_offset

            for col_idx, (cls, scale) in enumerate([
                (cls_a, 'shared'), (cls_a, 'per'),
                (cls_b, 'shared'), (cls_b, 'per')
            ]):
                ax = axes[row_idx, col_idx]
                mean_attr = mean_by_class[cls]

                if mean_attr is None:
                    ax.axis('off')
                    ax.set_title(f'{class_names[cls]} | no trials', fontsize=7)
                    continue

                vmax = global_vmax if scale == 'shared' else subj_vmax
                show_yticks = col_idx in [0, 2]

                sns.heatmap(
                    mean_attr, ax=ax,
                    cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                    xticklabels=False,
                    yticklabels=ch16_list if show_yticks else False,
                    cbar=True,
                    cbar_kws={'label': 'relevance', 'shrink': 0.8}
                )
                ax.set_xticks(tick_positions)
                ax.set_xticklabels(tick_labels, rotation=0, fontsize=7)
                ax.set_xlabel('time (s)', fontsize=8)

                if show_yticks:
                    ax.tick_params(axis='y', labelsize=7)

                if col_idx == 0 and row_offset == 0:
                    ax.set_title(
                        f'{subj} [{group}] | {class_names[cls]} | shared',
                        fontsize=8, loc='left'
                    )
                else:
                    ax.set_title(
                        f'{class_names[cls]} | '
                        f'{"shared" if scale == "shared" else "per-subject"}',
                        fontsize=8
                    )

    plt.suptitle(
        'LRP heatmaps — multiclass-16ch, all subjects',
        fontsize=12, y=1.001
    )
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / 'lrp_grouped_heatmaps_multiclass.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()

#### plot_grouped_bars_multiclass

In [ ]:
def plot_grouped_bars_multiclass(lrp_results, xai_subjects, save=True):
    """
    Channel importance bars for 4-class multiclass LRP results.
    26 rows (2 per subject), 4 columns.
    Row 1: left and right, Row 2: both-fists and both-feet.
    """
    class_names = ['left', 'right', 'both-fists', 'both-feet']
    grouped_order, subject_to_group = get_grouped_order(xai_subjects, lrp_results)
    n_subjects = len(grouped_order)

    global_xmax = max(
        np.abs(lrp_results[s][2][cls]).mean(axis=1).max()
        for s in grouped_order
        for cls in range(4)
        if lrp_results[s][2][cls] is not None
    )

    colors = ['#d62728' if ch in ['C3', 'Cz', 'C4'] else '#1f77b4'
              for ch in ch16_list]

    fig, axes = plt.subplots(
        n_subjects * 2, 4,
        figsize=(20, 2.5 * n_subjects * 2)
    )

    for subj_idx, subj in enumerate(grouped_order):
        _, _, mean_by_class = lrp_results[subj]
        group = subject_to_group[subj]

        for row_offset, (cls_a, cls_b) in enumerate([(0, 1), (2, 3)]):
            row_idx = subj_idx * 2 + row_offset

            for col_idx, (cls, scale) in enumerate([
                (cls_a, 'shared'), (cls_a, 'per'),
                (cls_b, 'shared'), (cls_b, 'per')
            ]):
                ax = axes[row_idx, col_idx]
                mean_attr = mean_by_class[cls]

                if mean_attr is None:
                    ax.axis('off')
                    continue

                mean_abs = np.abs(mean_attr).mean(axis=1)
                xmax = global_xmax if scale == 'shared' else mean_abs.max() * 1.1

                ax.barh(ch16_list, mean_abs, color=colors)
                ax.set_xlim(0, xmax)
                ax.invert_yaxis()
                ax.set_xlabel('mean |relevance|', fontsize=7)

                if col_idx in [1, 3]:
                    ax.set_yticklabels([])
                else:
                    ax.tick_params(axis='y', labelsize=7)

                if col_idx == 0 and row_offset == 0:
                    ax.set_title(
                        f'{subj} [{group}] | {class_names[cls]} | shared',
                        fontsize=8, loc='left'
                    )
                else:
                    ax.set_title(
                        f'{class_names[cls]} | '
                        f'{"shared" if scale == "shared" else "per-subject"}',
                        fontsize=8
                    )

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d62728', label='motor cortex (C3/Cz/C4)'),
        Patch(facecolor='#1f77b4', label='other channels')
    ]
    axes[0, 3].legend(handles=legend_elements, loc='lower right', fontsize=7)

    plt.suptitle(
        'LRP channel importance — multiclass-16ch, all subjects',
        fontsize=12, y=1.001
    )
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / 'lrp_grouped_bars_multiclass.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()

#### Missclasified trials check

In [ ]:
def plot_misclassified_combined(lrp_misc_results, xai_subjects, save=True):
    """
    Combined misclassification plot, one image.
    One row per subject, three columns:
    col 1: unilateral->bilateral mean attribution
    col 2: bilateral->unilateral mean attribution
    col 3: cross-unilateral mean attribution
    Shared colormap across all panels.
    Empty panel shown explicitly when no trials of that type exist.
    """
    grouped_order, subject_to_group = get_grouped_order(
        xai_subjects, lrp_misc_results
    )
    n_subjects = len(grouped_order)

    unilateral = {0, 1}
    bilateral = {2, 3}

    def get_mean_attr(misc_attrs, misc_true, misc_pred, true_set, pred_set):
        """Average attribution across all trials matching true/pred sets."""
        if len(misc_attrs) == 0:
            return None
        mask = (np.isin(misc_true, list(true_set)) &
                np.isin(misc_pred, list(pred_set)))
        if mask.sum() == 0:
            return None
        return misc_attrs[mask].mean(axis=0)

    # compute all mean attributions first to get global vmax
    all_maps = []
    subject_maps = {}
    for subj in grouped_order:
        misc_attrs, misc_true, misc_pred, _ = lrp_misc_results[subj]
        u2b = get_mean_attr(misc_attrs, misc_true, misc_pred, unilateral, bilateral)
        b2u = get_mean_attr(misc_attrs, misc_true, misc_pred, bilateral, unilateral)
        cu = get_mean_attr(misc_attrs, misc_true, misc_pred, unilateral, unilateral)
        subject_maps[subj] = (u2b, b2u, cu)
        for m in [u2b, b2u, cu]:
            if m is not None:
                all_maps.append(np.abs(m).max())

    global_vmax = max(all_maps) if all_maps else 0.04

    timepoints = 480
    tick_positions = list(range(0, timepoints, 80))
    tick_labels = [f"{t/160:.1f}" for t in tick_positions]

    col_titles = [
        'unilateral → bilateral',
        'bilateral → unilateral',
        'cross-unilateral (left ↔ right)'
    ]

    fig, axes = plt.subplots(n_subjects, 3, figsize=(18, 3.5 * n_subjects))

    for row_idx, subj in enumerate(grouped_order):
        group = subject_to_group[subj]
        u2b, b2u, cu = subject_maps[subj]

        for col_idx, mean_attr in enumerate([u2b, b2u, cu]):
            ax = axes[row_idx, col_idx]

            if mean_attr is None:
                ax.set_facecolor('#f5f5f5')
                ax.text(0.5, 0.5, 'no trials',
                        ha='center', va='center',
                        transform=ax.transAxes, fontsize=9, color='grey')
                ax.set_xticks([])
                ax.set_yticks([])
                if row_idx == 0:
                    ax.set_title(col_titles[col_idx], fontsize=9)
                continue

            sns.heatmap(
                mean_attr, ax=ax,
                cmap='RdBu_r', vmin=-global_vmax, vmax=global_vmax,
                xticklabels=False,
                yticklabels=ch16_list if col_idx == 0 else False,
                cbar=col_idx == 2,
                cbar_kws={'label': 'relevance', 'shrink': 0.8}
            )
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=0, fontsize=7)
            ax.set_xlabel('time (s)', fontsize=8)

            if col_idx == 0:
                ax.set_ylabel(f'{subj} [{group}]', fontsize=8)
                ax.tick_params(axis='y', labelsize=7)

            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontsize=9)

    plt.suptitle(
        'LRP misclassified trials — multiclass-16ch, error type comparison',
        fontsize=12, y=1.001
    )
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / 'lrp_misclassified_combined.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()

In [ ]:
def plot_misclassified_by_type(lrp_misc_results, xai_subjects,
                                error_type='u2b', save=True):
    """
    One figure per error type, grouped row ordering, shared and per-subject cols.
    error_type: 'u2b' (unilateral->bilateral),
                'b2u' (bilateral->unilateral),
                'cu'  (cross-unilateral)
    """
    type_config = {
        'u2b': {
            'true_set': {0, 1}, 'pred_set': {2, 3},
            'title': 'unilateral to bilateral',
            'filename': 'lrp_misclassified_u2b.png'
        },
        'b2u': {
            'true_set': {2, 3}, 'pred_set': {0, 1},
            'title': 'bilateral to unilateral',
            'filename': 'lrp_misclassified_b2u.png'
        },
        'cu': {
            'true_set': {0, 1}, 'pred_set': {0, 1},
            'title': 'cross-unilateral (left vs right)',
            'filename': 'lrp_misclassified_cu.png'
        }
    }
    cfg = type_config[error_type]
    grouped_order, subject_to_group = get_grouped_order(
        xai_subjects, lrp_misc_results
    )
    n_subjects = len(grouped_order)

    def get_mean_attr(misc_attrs, misc_true, misc_pred, true_set, pred_set):
        if len(misc_attrs) == 0:
            return None
        mask = (np.isin(misc_true, list(true_set)) &
                np.isin(misc_pred, list(pred_set)))
        # for cross-unilateral, exclude same-class predictions
        if true_set == pred_set:
            mask = mask & (misc_true != misc_pred)
        if mask.sum() == 0:
            return None
        return misc_attrs[mask].mean(axis=0)

    # compute maps and global scale
    subject_maps = {}
    all_max = []
    for subj in grouped_order:
        misc_attrs, misc_true, misc_pred, _ = lrp_misc_results[subj]
        m = get_mean_attr(misc_attrs, misc_true, misc_pred,
                          cfg['true_set'], cfg['pred_set'])
        subject_maps[subj] = m
        if m is not None:
            all_max.append(np.abs(m).max())

    global_vmax = max(all_max) if all_max else 0.04

    timepoints = 480
    tick_positions = list(range(0, timepoints, 80))
    tick_labels = [f"{t/160:.1f}" for t in tick_positions]

    fig, axes = plt.subplots(n_subjects, 2, figsize=(12, 3.5 * n_subjects))

    for row_idx, subj in enumerate(grouped_order):
        group = subject_to_group[subj]
        mean_attr = subject_maps[subj]

        for col_idx, scale in enumerate(['shared', 'per-subject']):
            ax = axes[row_idx, col_idx]

            if mean_attr is None:
                ax.set_facecolor('#f5f5f5')
                ax.text(0.5, 0.5, 'no trials',
                        ha='center', va='center',
                        transform=ax.transAxes, fontsize=9, color='grey')
                ax.set_xticks([])
                ax.set_yticks([])
                if col_idx == 0:
                    ax.set_ylabel(f'{subj} [{group}]', fontsize=8)
                continue

            vmax = global_vmax if scale == 'shared' else np.abs(mean_attr).max()

            sns.heatmap(
                mean_attr, ax=ax,
                cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                xticklabels=False,
                yticklabels=ch16_list if col_idx == 0 else False,
                cbar=True,
                cbar_kws={'label': 'relevance', 'shrink': 0.8}
            )
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=0, fontsize=7)
            ax.set_xlabel('time (s)', fontsize=8)

            if col_idx == 0:
                ax.set_ylabel(f'{subj} [{group}]', fontsize=8)
                ax.tick_params(axis='y', labelsize=7)

            if row_idx == 0:
                ax.set_title(f'{scale}', fontsize=9)

    plt.suptitle(
        f'LRP misclassified — {cfg["title"]} — multiclass-16ch',
        fontsize=12, y=1.001
    )
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / cfg['filename']
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()

#### plot_stopping_criterion_consistency

In [ ]:
consistency_subjects = ['S007', 'S085', 'S108', 'S107', 'S080',
                         'S097', 'S037', 'S006', 'S039', 'S018']

In [ ]:
def plot_stopping_criterion_consistency(lrp_kappa, lrp_composite,
                                         subjects, save=True):
    """
    Compares kappa vs composite stopping attribution for same subjects.
    One row per subject, 4 columns:
    left-kappa | left-composite | right-kappa | right-composite
    Shared colormap per subject (not global) so pattern is visible
    regardless of magnitude differences between criteria.
    """
    n_subjects = len(subjects)

    timepoints = 480
    tick_positions = list(range(0, timepoints, 80))
    tick_labels = [f"{t/160:.1f}" for t in tick_positions]

    fig, axes = plt.subplots(n_subjects, 4, figsize=(22, 3.5 * n_subjects))

    col_titles = [
        'left | kappa', 'left | composite',
        'right | kappa', 'right | composite'
    ]

    for row_idx, subj in enumerate(subjects):
        _, _, mean_kappa = lrp_kappa[subj]
        _, _, mean_comp = lrp_composite[subj]

        # per-subject vmax across both criteria and both classes
        # so kappa and composite are on same scale for fair comparison
        subj_vmax = max(
            np.abs(mean_kappa[cls]).max()
            for cls in [0, 1] if mean_kappa[cls] is not None
        )
        comp_vmax = max(
            np.abs(mean_comp[cls]).max()
            for cls in [0, 1] if mean_comp[cls] is not None
        )
        vmax = max(subj_vmax, comp_vmax)

        panels = [
            (0, mean_kappa[0]),
            (1, mean_comp[0]),
            (2, mean_kappa[1]),
            (3, mean_comp[1]),
        ]

        for col_idx, mean_attr in panels:
            ax = axes[row_idx, col_idx]

            if mean_attr is None:
                ax.set_facecolor('#f5f5f5')
                ax.text(0.5, 0.5, 'no trials',
                        ha='center', va='center',
                        transform=ax.transAxes, fontsize=9, color='grey')
                ax.set_xticks([])
                ax.set_yticks([])
                continue

            show_yticks = col_idx == 0
            sns.heatmap(
                mean_attr, ax=ax,
                cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                xticklabels=False,
                yticklabels=ch16_list if show_yticks else False,
                cbar=col_idx == 3,
                cbar_kws={'label': 'relevance', 'shrink': 0.8}
            )
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=0, fontsize=7)
            ax.set_xlabel('time (s)', fontsize=8)

            if show_yticks:
                ax.set_ylabel(subj, fontsize=9)
                ax.tick_params(axis='y', labelsize=7)

            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontsize=9)

    plt.suptitle(
        'LRP stopping-criterion consistency — kappa vs composite — binary CDAN-16ch',
        fontsize=12, y=1.001
    )
    plt.tight_layout()

    if save:
        save_path = results_path / 'figures' / 'lrp_stopping_consistency.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()

#### data save

In [ ]:
## Folder creation for data
lrp_attr_path = results_path / "lrp_attributions"
binary_attr_path = lrp_attr_path / "binary_cdan_16ch"
multiclass_attr_path = lrp_attr_path / "multiclass_16ch"

binary_attr_path.mkdir(parents=True, exist_ok=True)
multiclass_attr_path.mkdir(parents=True, exist_ok=True)

# print("Folders created:")
# print(f"  {binary_attr_path}")
# print(f"  {multiclass_attr_path}")

In [ ]:
def run_lrp_full(valid_subjects, checkpoint_tag, attr_save_path,
                 n_classes=2, use_multiclass=False, first_layer_rule='epsilon'):
    """
    Runs LRP on all valid subjects, saves per-subject attributions to disk.
    Resume-capable: skips subjects with existing .npz files.
    Saves per subject: mean attribution per class, labels, trial count.
    # both kappa and composite stopping criteria saved per subject.
    # one-time run makes storage cost negligible relative to rerun cost.
    # composite may be needed for stopping-criterion comparison in thesis
    # writing or if a reviewer asks about criterion sensitivity.
    """
    attr_save_path = Path(attr_save_path)
    completed = []
    skipped = []
    failed = []

    for idx, subj in enumerate(valid_subjects):
        save_file = attr_save_path / f"{subj}_lrp.npz"

        if save_file.exists():
            skipped.append(subj)
            continue

        try:
            # run both stopping criteria, save together in one file
            # one-time run, cheap to save both, avoids rerunning if
            # composite attributions are needed for thesis writing or
            # stopping-criterion comparison figures
            save_dict = {'n_classes': n_classes}

            for criterion in ['kappa', 'composite']:
                attrs, labels, mean_by_class = compute_lrp(
                    subject=subj,
                    checkpoint_tag=checkpoint_tag,
                    n_classes=n_classes,
                    use_multiclass=use_multiclass,
                    stopping_criterion=criterion,
                    first_layer_rule=first_layer_rule,
                    trial_type='correct',
                )
                save_dict[f'labels_{criterion}'] = labels
                save_dict[f'n_trials_{criterion}'] = len(attrs)
                for cls in range(n_classes):
                    if mean_by_class[cls] is not None:
                        save_dict[f'mean_attr_class_{cls}_{criterion}'] = mean_by_class[cls]

            np.savez(save_file, **save_dict)
            completed.append(subj)

            if (idx + 1) % 10 == 0:
                print(f"Progress: {idx+1}/{len(valid_subjects)} "
                      f"({len(completed)} computed, "
                      f"{len(skipped)} skipped, "
                      f"{len(failed)} failed)")

        except Exception as e:
            print(f"FAILED {subj}: {e}")
            failed.append(subj)

    print(f"\nDone. Computed: {len(completed)}, "
          f"Skipped: {len(skipped)}, Failed: {len(failed)}")
    if failed:
        print(f"Failed subjects: {failed}")

    return completed, skipped, failed

In [ ]:
def load_lrp_attributions(attr_save_path, valid_subjects, n_classes=2):
    """
    Loads all saved LRP attribution files into memory.
    Returns dict: {subject_id: {criterion: {class_idx: mean_attr}}}
    """
    attr_save_path = Path(attr_save_path)
    all_results = {}
    missing = []

    for subj in valid_subjects:
        save_file = attr_save_path / f"{subj}_lrp.npz"
        if not save_file.exists():
            missing.append(subj)
            continue

        data = np.load(save_file, allow_pickle=False)
        subj_results = {}

        for criterion in ['kappa', 'composite']:
            mean_by_class = {}
            for cls in range(n_classes):
                key = f'mean_attr_class_{cls}_{criterion}'
                mean_by_class[cls] = data[key] if key in data else None
            subj_results[criterion] = {
                'mean_by_class': mean_by_class,
                'n_trials': int(data[f'n_trials_{criterion}']),
            }

        all_results[subj] = subj_results

    print(f"Loaded: {len(all_results)}/{ len(valid_subjects)} subjects")
    if missing:
        print(f"Missing: {missing}")

    return all_results

#### Per class accuracy

In [ ]:
def compute_per_class_accuracy_106(valid_subjects, checkpoint_tag,
                                    save_path, n_classes=4,
                                    use_multiclass=True, use_16ch=True,
                                    stopping_criterion='kappa'):
    """
    Loads each subject's checkpoint, runs evaluate(), saves per-class
    accuracy and confusion matrix for all 106 subjects.
    Resume-capable: skips if output file already exists.
    """
    save_file = Path(save_path) / f"{checkpoint_tag}_per_class_accuracy.json"

    if save_file.exists():
        print(f"Already exists, loading: {save_file}")
        with open(save_file) as f:
            return json.load(f)

    results = {}
    failed = []

    for idx, subj in enumerate(valid_subjects):
        try:
            # load target data once, reused for both criteria
            _, _, target_eval_loader, _ = prepare_dataloaders(
                test_subject_id=subj, valid_subjects=valid_subjects,
                use_16ch=use_16ch, use_multiclass=use_multiclass
            )

            results[subj] = {}

            for criterion in ['kappa', 'composite']:
                ckpt_path = (results_path / "checkpoints" / checkpoint_tag /
                            f"{subj}_{criterion}.pt")
                state_dict = torch.load(ckpt_path, map_location="cpu",
                                       weights_only=False)

                model = DANN(
                    n_channels=16, n_timepoints=480, n_classes=n_classes,
                    embed_dim=128, dropout_rate=0.25,
                    use_conditional_discriminator=True,
                    feature_extractor_type='eegnet'
                )
                model.load_state_dict(state_dict)
                model.eval()

                acc, kappa, auc, per_class_acc, conf_mat = evaluate(
                    model, target_eval_loader
                )

                results[subj][criterion] = {
                    'kappa': kappa,
                    'accuracy': acc,
                    'per_class_accuracy': per_class_acc,
                    'confusion_matrix': conf_mat,
                }

            if (idx + 1) % 10 == 0:
                print(f"Progress: {idx+1}/{len(valid_subjects)}")

        except Exception as e:
            print(f"FAILED {subj}: {e}")
            failed.append(subj)

    with open(save_file, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"\nDone. {len(results)}/{ len(valid_subjects)} subjects saved.")
    print(f"Saved to: {save_file}")
    if failed:
        print(f"Failed: {failed}")

    return results

#### classify_failure_groups - multiclass

In [ ]:
def classify_failure_groups(all_per_class, all_kappas,
                             spread_threshold=0.4,
                             min_acc_threshold=0.2,
                             flat_spread_threshold=0.2,
                             flat_kappa_threshold=0.3):
    """
    Classifies subjects into failure groups based on per-class accuracy.

    Group A (selective collapse): spread > spread_threshold AND
        min class accuracy < min_acc_threshold.
        One class is substantially worse than others.

    Group B (flat near-chance): spread < flat_spread_threshold AND
        overall kappa < flat_kappa_threshold.
        All classes performing similarly poorly.

    Neither: everything else.

    Thresholds derived from full 106-subject distribution:
        spread mean=0.343, std=0.145
        min_acc mean=0.332, std=0.165
    """
    group_a = []
    group_b = []
    neither = []

    for subj in all_per_class:
        pca = all_per_class[subj]
        spread = max(pca.values()) - min(pca.values())
        min_acc = min(pca.values())
        kappa = all_kappas[subj]
        collapsed_class = min(pca, key=pca.get)

        if kappa < 0.1:
            neither.append(subj)

        elif spread > spread_threshold and min_acc < min_acc_threshold:
            group_a.append({
                'subject': subj,
                'spread': round(spread, 3),
                'min_acc': round(min_acc, 3),
                'collapsed_class': collapsed_class,
                'kappa': round(kappa, 4),
            })
        elif spread < flat_spread_threshold and kappa < flat_kappa_threshold:
            group_b.append({
                'subject': subj,
                'spread': round(spread, 3),
                'min_acc': round(min_acc, 3),
                'kappa': round(kappa, 4),
            })
        else:
            neither.append(subj)

    return group_a, group_b, neither

#### plot_group_mean_attribution

In [ ]:
def plot_group_mean_attribution(binary_attrs_106, category_groups_106,
                                 score_groups, valid_subjects,
                                 all_kappas, sfreq=160, save=True):
    """
    Group-level mean attribution for all 106 subjects.
    Two figures: category groups and score groups.
    Rows = groups, columns = classes (left, right for binary).
    Each cell = mean attribution averaged across all subjects in that group.
    """
    class_names = ['left', 'right']

    # category group order and labels
    cat_group_order = ['negative_transfer', 'group_a_collapse',
                       'group_b_flat', 'neither']
    cat_group_labels = {
        'negative_transfer': f'negative transfer (n={sum(1 for s in valid_subjects if category_groups_106[s]=="negative_transfer")})',
        'group_a_collapse': f'group A collapse (n={sum(1 for s in valid_subjects if category_groups_106[s]=="group_a_collapse")})',
        'group_b_flat': f'group B flat (n={sum(1 for s in valid_subjects if category_groups_106[s]=="group_b_flat")})',
        'neither': f'neither (n={sum(1 for s in valid_subjects if category_groups_106[s]=="neither")})',
    }

    # score group order
    score_group_order = ['high', 'mid_high', 'mid_low', 'low',
                          'negative_transfer']
    score_group_labels = {
        g: f'{g} (n={sum(1 for s in valid_subjects if score_groups[s]==g)})'
        for g in score_group_order
    }

    timepoints = 480
    tick_positions = list(range(0, timepoints, 80))
    tick_labels = [f"{t/sfreq:.1f}" for t in tick_positions]

    for grouping_name, group_order, group_labels, group_dict in [
        ('category', cat_group_order, cat_group_labels, category_groups_106),
        ('score', score_group_order, score_group_labels, score_groups)
    ]:
        n_groups = len(group_order)
        fig, axes = plt.subplots(
            n_groups, 2, figsize=(14, 3.5 * n_groups)
        )

        # compute global vmax across all groups and classes
        all_means = []
        group_mean_maps = {}
        for grp in group_order:
            group_mean_maps[grp] = {}
            for cls in [0, 1]:
                members = [
                    s for s in valid_subjects
                    if group_dict[s] == grp
                    and binary_attrs_106[s]['kappa']['mean_by_class'][cls] is not None
                ]
                if members:
                    stacked = np.array([
                        binary_attrs_106[s]['kappa']['mean_by_class'][cls]
                        for s in members
                    ])
                    group_mean_maps[grp][cls] = stacked.mean(axis=0)
                    all_means.append(np.abs(group_mean_maps[grp][cls]).max())
                else:
                    group_mean_maps[grp][cls] = None

        global_vmax = max(all_means) if all_means else 0.04

        for row_idx, grp in enumerate(group_order):
            for col_idx, cls in enumerate([0, 1]):
                ax = axes[row_idx, col_idx]
                mean_attr = group_mean_maps[grp][cls]

                if mean_attr is None:
                    ax.set_facecolor('#f5f5f5')
                    ax.text(0.5, 0.5, 'no subjects',
                            ha='center', va='center',
                            transform=ax.transAxes, fontsize=9)
                    ax.set_xticks([])
                    ax.set_yticks([])
                    continue

                sns.heatmap(
                    mean_attr, ax=ax,
                    cmap='RdBu_r',
                    vmin=-global_vmax, vmax=global_vmax,
                    xticklabels=False,
                    yticklabels=ch16_list if col_idx == 0 else False,
                    cbar=col_idx == 1,
                    cbar_kws={'label': 'relevance', 'shrink': 0.8}
                )
                ax.set_xticks(tick_positions)
                ax.set_xticklabels(tick_labels, rotation=0, fontsize=7)
                ax.set_xlabel('time (s)', fontsize=8)

                if col_idx == 0:
                    ax.set_ylabel(group_labels[grp], fontsize=8)
                    ax.tick_params(axis='y', labelsize=7)

                if row_idx == 0:
                    ax.set_title(f'class: {class_names[cls]}', fontsize=9)

        plt.suptitle(
            f'LRP group-level mean attribution — binary CDAN-16ch '
            f'({grouping_name} grouping)',
            fontsize=12, y=1.001
        )
        plt.tight_layout()

        if save:
            save_path = (results_path / 'figures' /
                        f'lrp_group_mean_{grouping_name}_binary.png')
            plt.savefig(save_path, dpi=120, bbox_inches='tight')
            print(f"Saved: {save_path}")
        plt.show()

#### compute_within_group_consistency

In [ ]:
def compute_within_group_consistency(attrs_106, group_assignments,
                                      valid_subjects, grouping='category',
                                      config='binary', criterion='kappa',
                                      n_classes=2):
    """
    Computes within-group attribution consistency using pairwise correlation.

    Parameters:
    -----------
    attrs_106: dict, binary_attrs_106 or mc_attrs_106
    group_assignments: dict, category_groups_106 or score_groups
    valid_subjects: list of all valid subject IDs
    grouping: 'category' or 'score', which grouping to use
    config: 'binary' (n_classes=2) or 'multiclass' (n_classes=4)
    criterion: 'kappa' or 'composite'
    n_classes: 2 for binary, 4 for multiclass

    Returns:
    --------
    consistency_results: dict {group_name: {class_idx: mean_correlation}}
    """
    # get unique groups
    groups = sorted(set(group_assignments.values()))
    consistency_results = {}

    for grp in groups:
        grp_subjects = [
            s for s in valid_subjects
            if group_assignments[s] == grp
            and s in attrs_106
        ]

        if len(grp_subjects) < 2:
            consistency_results[grp] = None
            continue

        consistency_results[grp] = {}

        for cls in range(n_classes):
            # collect attribution maps for this class across group members
            maps = []
            for subj in grp_subjects:
                mean_attr = attrs_106[subj][criterion]['mean_by_class'].get(cls)
                if mean_attr is not None:
                    maps.append(mean_attr.flatten())

            if len(maps) < 2:
                consistency_results[grp][cls] = None
                continue

            # pairwise correlation across all subject pairs
            maps = np.array(maps)
            n = len(maps)
            correlations = []
            for i in range(n):
                for j in range(i+1, n):
                    r = np.corrcoef(maps[i], maps[j])[0, 1]
                    correlations.append(r)

            consistency_results[grp][cls] = {
                'mean_r': round(float(np.mean(correlations)), 4),
                'std_r': round(float(np.std(correlations)), 4),
                'n_subjects': n,
                'n_pairs': len(correlations),
            }

    return consistency_results


def print_consistency_results(consistency_results, n_classes=2,
                               class_names=None):
    """Prints within-group consistency results in readable format."""
    if class_names is None:
        class_names = ['left', 'right'] if n_classes == 2 else \
                      ['left', 'right', 'both-fists', 'both-feet']

    for grp, grp_results in consistency_results.items():
        if grp_results is None:
            print(f"\n{grp}: insufficient subjects (<2), skipped")
            continue
        print(f"\n{grp}:")
        for cls in range(n_classes):
            r = grp_results.get(cls)
            if r is None:
                print(f"  class {class_names[cls]}: no data")
            else:
                print(f"  class {class_names[cls]}: "
                      f"mean_r={r['mean_r']:.4f} "
                      f"(std={r['std_r']:.4f}, "
                      f"n={r['n_subjects']} subjects, "
                      f"{r['n_pairs']} pairs)")

#### compute_between_group_contrast

In [ ]:
def compute_between_group_contrast(attrs_106, group_assignments,
                                    valid_subjects, criterion='kappa',
                                    n_classes=2):
    """
    Computes between-group contrast by correlating group-level mean
    attribution maps against each other.
    Low correlation = groups look genuinely different.
    High correlation = groups look similar despite different categories.
    """
    groups = sorted(set(group_assignments.values()))

    # compute group mean maps
    group_means = {}
    for grp in groups:
        grp_subjects = [
            s for s in valid_subjects
            if group_assignments[s] == grp
            and s in attrs_106
        ]
        if len(grp_subjects) == 0:
            continue
        group_means[grp] = {}
        for cls in range(n_classes):
            maps = [
                attrs_106[s][criterion]['mean_by_class'][cls]
                for s in grp_subjects
                if attrs_106[s][criterion]['mean_by_class'].get(cls) is not None
            ]
            if maps:
                group_means[grp][cls] = np.array(maps).mean(axis=0)
            else:
                group_means[grp][cls] = None

    # pairwise correlation between group means
    group_list = list(group_means.keys())
    results = {}

    for cls in range(n_classes):
        results[cls] = {}
        for i, grp_a in enumerate(group_list):
            for grp_b in group_list[i+1:]:
                map_a = group_means[grp_a][cls]
                map_b = group_means[grp_b][cls]
                if map_a is None or map_b is None:
                    results[cls][f'{grp_a}_vs_{grp_b}'] = None
                    continue
                r = np.corrcoef(map_a.flatten(), map_b.flatten())[0, 1]
                results[cls][f'{grp_a}_vs_{grp_b}'] = round(float(r), 4)

    return results, group_means


def print_between_group_results(results, n_classes=2, class_names=None):
    if class_names is None:
        class_names = ['left', 'right'] if n_classes == 2 else \
                      ['left', 'right', 'both-fists', 'both-feet']

    for cls in range(n_classes):
        print(f"\nClass: {class_names[cls]}")
        for pair, r in results[cls].items():
            if r is None:
                print(f"  {pair}: no data")
            else:
                print(f"  {pair}: r={r:.4f}")

### compute_attribution_performance_correlation

In [ ]:
def compute_attribution_performance_correlation(attrs_106, group_assignments,
                                                 all_kappas, valid_subjects,
                                                 target_group='neither',
                                                 target_channels=None,
                                                 criterion='kappa',
                                                 n_classes=2):
    """
    Correlates mean absolute relevance on target channels with kappa score.
    Computed within a specific group to avoid confounding by group membership.

    target_channels: list of channel names to measure relevance on.
                     Default: ['C3', 'C4', 'CP3']
    """
    if target_channels is None:
        target_channels = ['C3', 'C4', 'CP3']

    # get channel indices
    channel_indices = [ch16_list.index(ch) for ch in target_channels
                       if ch in ch16_list]

    group_subjects = [
        s for s in valid_subjects
        if group_assignments[s] == target_group
        and s in attrs_106
    ]

    print(f"Group '{target_group}': {len(group_subjects)} subjects")

    results = {}
    for cls in range(n_classes):
        motor_relevance = []
        kappas = []

        for subj in group_subjects:
            mean_attr = attrs_106[subj][criterion]['mean_by_class'].get(cls)
            if mean_attr is None:
                continue
            # mean absolute relevance on target channels, collapsed across time
            channel_relevance = np.abs(mean_attr[channel_indices, :]).mean()
            motor_relevance.append(float(channel_relevance))
            kappas.append(all_kappas[subj])

        if len(motor_relevance) < 3:
            results[cls] = None
            continue

        r = np.corrcoef(motor_relevance, kappas)[0, 1]
        results[cls] = {
            'r': round(float(r), 4),
            'n': len(motor_relevance),
            'channels': target_channels,
        }

    return results

#### compute_within_group_performance_drivers

In [ ]:
def compute_within_group_performance_drivers(attrs_106, group_assignments,
                                              all_kappas, valid_subjects,
                                              target_group='neither',
                                              criterion='kappa',
                                              n_classes=2,
                                              motor_channels=None):
    if motor_channels is None:
        motor_channels = ['C3', 'C4', 'CP3']

    channel_indices = [ch16_list.index(ch) for ch in motor_channels]
    window_start = int(0.3 * 160)
    window_end = int(1.5 * 160)

    group_subjects = [
        s for s in valid_subjects
        if group_assignments[s] == target_group
        and s in attrs_106
    ]

    print(f"Found {len(group_subjects)} subjects in group '{target_group}'")

    if len(group_subjects) == 0:
        print("No subjects found, returning None")
        return None

    trial_counts = []
    magnitudes = []
    temporal_concentrations = []
    kappas = []

    for subj in group_subjects:
        subj_data = attrs_106[subj][criterion]
        n_trials = subj_data['n_trials']

        subj_magnitudes = []
        subj_concentrations = []

        for cls in range(n_classes):
            mean_attr = subj_data['mean_by_class'].get(cls)
            if mean_attr is None:
                continue
            mag = float(np.abs(mean_attr).mean())
            subj_magnitudes.append(mag)
            total_relevance = np.abs(mean_attr).sum()
            window_relevance = np.abs(
                mean_attr[:, window_start:window_end]
            ).sum()
            if total_relevance > 0:
                concentration = float(window_relevance / total_relevance)
                subj_concentrations.append(concentration)

        trial_counts.append(n_trials)
        magnitudes.append(
            np.mean(subj_magnitudes) if subj_magnitudes else 0
        )
        temporal_concentrations.append(
            np.mean(subj_concentrations) if subj_concentrations else 0
        )
        kappas.append(all_kappas[subj])

    trial_counts = np.array(trial_counts)
    magnitudes = np.array(magnitudes)
    temporal_concentrations = np.array(temporal_concentrations)
    kappas = np.array(kappas)

    r_trials = np.corrcoef(trial_counts, kappas)[0, 1]
    r_magnitude = np.corrcoef(magnitudes, kappas)[0, 1]
    r_temporal = np.corrcoef(temporal_concentrations, kappas)[0, 1]

    print("Computation complete, returning results")
    return {
        'trial_count_vs_kappa': round(float(r_trials), 4),
        'magnitude_vs_kappa': round(float(r_magnitude), 4),
        'temporal_concentration_vs_kappa': round(float(r_temporal), 4),
        'n_subjects': len(group_subjects),
    }

#### End of functions

In [ ]:
plot_grouped_bars(lrp_results, xai_subjects)
plot_grouped_heatmaps(lrp_results, xai_subjects)

In [ ]:
# load one subject directly from the 16ch processed folder
sample = np.load(ch16_path / 'S007_16ch.npz')
# print("Keys in file:", list(sample.keys()))

# # check if channel names are stored
# if 'channel_names' in sample:
#     print("Stored channel names:", sample['channel_names'])
# else:
#     print("No channel names stored in file")

# check data shape to confirm channel count and order assumption
# sample = np.load(ch16_path / 'S007_16ch.npz')
print("Keys in file:", list(sample.keys()))
print("Data shape:", sample['data'].shape)
print("Labels shape:", sample['labels'].shape)
print("Unique labels:", np.unique(sample['labels']))

#### Check channels

In [ ]:
import mne

sample_edf_path = Path("../Data/raw/physionet_mi_data/eeg-motor-movementimagery-dataset/S007/S007R04.edf")
raw = mne.io.read_raw_edf(sample_edf_path, preload=False, verbose=False)

# check MNE version behaviour
print("MNE version:", mne.__version__)

# what order does MNE store these channels in the raw file?
raw_ch_names = raw.ch_names
selected = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
            'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

# find where each selected channel sits in MNE's raw order
print("\nChannel positions in raw MNE order:")
for ch in selected:
    # strip dots since raw file may have them
    matches = [i for i, c in enumerate(raw_ch_names)
               if c.rstrip('.').upper() == ch.upper()]
    print(f"  {ch}: raw index {matches[0] if matches else 'NOT FOUND'}")

In [ ]:
selected_channels = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
                      'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']
raw = mne.io.read_raw_edf(sample_edf_path, preload=True, verbose=False)
from mne import events_from_annotations, Epochs

raw.notch_filter(freqs=60, verbose=False)
raw.filter(l_freq=8.0, h_freq=30.0, verbose=False)
raw.set_eeg_reference('average', projection=False, verbose=False)

# rename same as preprocessing
montage = mne.channels.make_standard_montage('standard_1020')
montage_lookup = {ch.upper(): ch for ch in montage.ch_names}
rename_map = {ch: ch.rstrip('.') for ch in raw.ch_names}
raw.rename_channels(rename_map)
cap_fix = {}
for ch in raw.ch_names:
    upper = ch.upper()
    cap_fix[ch] = montage_lookup[upper] if upper in montage_lookup else ch
raw.rename_channels(cap_fix)

events, _ = events_from_annotations(raw, verbose=False)
epochs = Epochs(raw, events, event_id={'T1': 2, 'T2': 3},
                tmin=0, tmax=(480-1)/160, baseline=None,
                preload=True, verbose=False)

# apply same pick_channels call as preprocessing
epochs.pick_channels(selected_channels)

print("Channel order AFTER pick_channels:")
for i, ch in enumerate(epochs.ch_names):
    print(f"  index {i}: {ch}")

print("\nDoes it match selected_channels?", epochs.ch_names == selected_channels)

#### composite batch compute

In [ ]:
lrp_results_composite = {}

for subj in xai_subset:
    print(f"Computing {subj}...")
    attrs, labels, mean_by_class = compute_lrp(
        subject=subj,
        checkpoint_tag='cdan_16ch_binary',
        n_classes=2,
        use_multiclass=False,
        stopping_criterion='composite',
        first_layer_rule='epsilon',
    )
    lrp_results_composite[subj] = (attrs, labels, mean_by_class)
    print(f"  {len(attrs)} correctly classified trials")

print(f"\nDone: {list(lrp_results_composite.keys())}")

#### Misclassified trials run test

##### S018 test

In [ ]:
misc_out = compute_lrp(
    subject='S018',
    checkpoint_tag='cdan_16ch_binary',
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    first_layer_rule='epsilon',
    trial_type='misclassified',
)

misc_attrs, misc_true, misc_pred, misc_mean = misc_out
print(f"Misclassified trials: {len(misc_attrs)}")
print(f"True labels: {misc_true}")
print(f"Predicted as: {misc_pred}")

In [ ]:
# S018 correct trials already in lrp_results
_, _, correct_mean_s018 = lrp_results['S018']

# misclassified already computed above as misc_out
misc_attrs_s018, misc_true_s018, misc_pred_s018, misc_mean_s018 = misc_out

class_names = ['left', 'right']

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# shared vmax across all four panels for fair comparison
vmax = max(
    max(np.abs(v).max() for v in correct_mean_s018.values() if v is not None),
    max(np.abs(v).max() for v in misc_mean_s018.values() if v is not None)
)

timepoints = 480
tick_positions = list(range(0, timepoints, 80))
tick_labels = [f"{t/160:.1f}" for t in tick_positions]

panels = [
    (0, 0, correct_mean_s018[0], 'correct | true: left'),
    (0, 1, correct_mean_s018[1], 'correct | true: right'),
    (1, 0, misc_mean_s018[0],    'misclassified | predicted: left (actually right)'),
    (1, 1, misc_mean_s018[1],    'misclassified | predicted: right (actually left)'),
]

for row, col, mean_attr, title in panels:
    ax = axes[row, col]
    if mean_attr is None:
        ax.axis('off')
        ax.set_title(f'S018 | {title}\nno trials', fontsize=9)
        continue
    sns.heatmap(
        mean_attr, ax=ax,
        cmap='RdBu_r', vmin=-vmax, vmax=vmax,
        xticklabels=False, yticklabels=ch16_list,
        cbar_kws={'label': 'relevance'}
    )
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=0, fontsize=9)
    ax.set_xlabel('time (s)', fontsize=10)
    ax.set_ylabel('channel', fontsize=10)
    ax.set_title(f'S018 | {title}', fontsize=10)

plt.suptitle('S018 correct vs misclassified attribution — binary CDAN-16ch',
             fontsize=12)
plt.tight_layout()
plt.savefig(results_path / 'figures' / 'lrp_S018_correct_vs_misclassified.png',
            dpi=150, bbox_inches='tight')
plt.show()

#### Score based grouping

In [ ]:
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    cdan_results = json.load(f)

print("Kappa scores for XAI subset:")
for subj in xai_subset:
    k = cdan_results[subj]['kappa_stopping']['final_kappa']
    print(f"  {subj}: {k:.4f}")

In [ ]:
all_kappas = {
    subj: cdan_results[subj]['kappa_stopping']['final_kappa']
    for subj in valid_subjects
}

below_chance = [(s, k) for s, k in all_kappas.items() if k < 0.1]
above_chance = [(s, k) for s, k in all_kappas.items() if k >= 0.1]

print(f"Below chance (kappa < 0.1): {len(below_chance)} subjects")
print(f"Above chance (kappa >= 0.1): {len(above_chance)} subjects")
print(f"\nBelow chance subjects: {sorted(below_chance, key=lambda x: x[1])}")

In [ ]:
all_kappas = {
    subj: cdan_results[subj]['kappa_stopping']['final_kappa']
    for subj in valid_subjects
}

# split on existing above-chance threshold
below_chance = {s: k for s, k in all_kappas.items() if k < 0.1}
above_chance = {s: k for s, k in all_kappas.items() if k >= 0.1}

# tertiles computed from above-chance subjects only
above_kappas = np.array(list(above_chance.values()))
t33 = np.percentile(above_kappas, 33.33)
t67 = np.percentile(above_kappas, 66.67)

print(f"Tertile boundaries (from {len(above_chance)} above-chance subjects):")
print(f"  Low-mid boundary: {t33:.4f}")
print(f"  Mid-high boundary: {t67:.4f}")

def assign_score_group(kappa, t33, t67):
    if kappa < 0.1:
        return 'negative_transfer'
    elif kappa < t33:
        return 'low'
    elif kappa < t67:
        return 'mid'
    else:
        return 'high'

score_groups = {
    subj: assign_score_group(k, t33, t67)
    for subj, k in all_kappas.items()
}

# check distribution
from collections import Counter
dist = Counter(score_groups.values())
print(f"\nFull 106-subject distribution: {dict(dist)}")

# apply to XAI subset
print(f"\nXAI subset score groups:")
xai_score_groups = {}
for subj in xai_subset:
    group = score_groups[subj]
    kappa = all_kappas[subj]
    xai_score_groups[subj] = group
    print(f"  {subj}: {kappa:.4f} -> {group}")

In [ ]:


# all 106 kappa values
kappa_values = np.array(list(all_kappas.values()))
mean_k = kappa_values.mean()
std_k = kappa_values.std()

# std bands
print("=== Standard deviation bands ===")
print(f"Mean: {mean_k:.4f}, Std: {std_k:.4f}")
print(f"High: > {mean_k + std_k:.4f}")
print(f"Mid: {mean_k - std_k:.4f} to {mean_k + std_k:.4f}")
print(f"Low: 0.1 to {mean_k - std_k:.4f}")
print(f"Negative: < 0.1")

sd_groups = {}
for s, k in all_kappas.items():
    if k < 0.1:
        sd_groups[s] = 'negative'
    elif k < mean_k - std_k:
        sd_groups[s] = 'low'
    elif k < mean_k + std_k:
        sd_groups[s] = 'mid'
    else:
        sd_groups[s] = 'high'

from collections import Counter
print(f"Distribution: {dict(Counter(sd_groups.values()))}")

# kernel density natural breaks
print("\n=== Kernel density natural breaks ===")
kde = gaussian_kde(kappa_values, bw_method=0.2)
x = np.linspace(kappa_values.min(), kappa_values.max(), 1000)
density = kde(x)

# find local minima in density as natural break points
from scipy.signal import argrelmin
minima_idx = argrelmin(density, order=30)[0]
break_points = x[minima_idx]
print(f"Natural breaks found at: {[f'{b:.4f}' for b in break_points]}")

# XAI subset under both
print("\n=== XAI subset comparison ===")
print(f"{'Subject':<8} {'Kappa':<8} {'SD group':<12} {'KDE break'}")
for subj in xai_subset:
    k = all_kappas[subj]
    sd_g = sd_groups[subj]
    # kde group based on breaks
    kde_g = 'group_1'
    for bp in sorted(break_points):
        if k > bp:
            kde_g = f'above_{bp:.2f}'
    print(f"{subj:<8} {k:<8.4f} {sd_g:<12} {kde_g}")

In [ ]:
multipliers = np.arange(0.2, 1.6, 0.1)
results = []

for mult in multipliers:
    high_boundary = mean_k + (mult * std_k)
    low_boundary = mean_k - (mult * std_k)

    groups = {}
    for s, k in all_kappas.items():
        if k < 0.1:
            groups[s] = 'negative'
        elif k < low_boundary:
            groups[s] = 'low'
        elif k < mean_k:
            groups[s] = 'mid_low'
        elif k < high_boundary:
            groups[s] = 'mid_high'
        else:
            groups[s] = 'high'

    # within-group kappa variance for each group
    group_variances = {}
    for grp in ['negative', 'low', 'mid_low', 'mid_high', 'high']:
        members = [all_kappas[s] for s, g in groups.items() if g == grp]
        group_variances[grp] = np.var(members) if len(members) > 1 else 0.0

    mean_within_variance = np.mean(list(group_variances.values()))

    # group sizes
    counts = Counter(groups.values())

    # xai subset spread, how many unique groups do the 13 subjects cover
    xai_groups = set(groups[s] for s in xai_subset)

    results.append({
        'multiplier': round(mult, 1),
        'low_boundary': round(low_boundary, 4),
        'high_boundary': round(high_boundary, 4),
        'mean_within_var': round(mean_within_variance, 6),
        'negative': counts.get('negative', 0),
        'low': counts.get('low', 0),
        'mid_low': counts.get('mid_low', 0),
        'mid_high': counts.get('mid_high', 0),
        'high': counts.get('high', 0),
        'xai_groups_covered': len(xai_groups),
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nBest multiplier by within-group variance: "
      f"{df.loc[df['mean_within_var'].idxmin(), 'multiplier']}")
print(f"Best multiplier with full XAI coverage (5 groups): "
      f"{df[df['xai_groups_covered']==5]['multiplier'].values}")

In [ ]:
# assign groups from full 106-subject kappa distribution
score_groups, boundaries = assign_score_groups(all_kappas, std_multiplier=0.7)

print("Boundaries:", boundaries)
print("\nXAI subset assignments:")
for subj in xai_subset:
    print(f"  {subj}: κ={all_kappas[subj]:.4f} -> {score_groups[subj]}")

# plots
plot_score_grouped_bars(lrp_results, score_groups, xai_subset,
                        boundaries, kappas_dict=all_kappas)
plot_score_grouped_heatmaps(lrp_results, score_groups, xai_subset,
                            boundaries, kappas_dict=all_kappas)

#### Multi class compute

In [ ]:
lrp_results_multiclass = {}

for subj in xai_subset:
    print(f"Computing {subj}...")
    attrs, labels, mean_by_class = compute_lrp(
        subject=subj,
        checkpoint_tag='multiclass_16ch',
        n_classes=4,
        use_multiclass=True,
        stopping_criterion='kappa',
        first_layer_rule='epsilon',
        trial_type='correct',
    )
    lrp_results_multiclass[subj] = (attrs, labels, mean_by_class)
    print(f"  {len(attrs)} correctly classified trials")

print(f"\nDone: {list(lrp_results_multiclass.keys())}")

In [ ]:
class_names_4 = ['left', 'right', 'both-fists', 'both-feet']

plot_grouped_bars(lrp_results_multiclass, xai_subjects,
                  class_names=class_names_4, save=True)
plot_grouped_heatmaps(lrp_results_multiclass, xai_subjects,
                      class_names=class_names_4, save=True)

In [ ]:
plot_grouped_bars_multiclass(lrp_results_multiclass, xai_subjects)
plot_grouped_heatmaps_multiclass(lrp_results_multiclass, xai_subjects)

#### Misclassified trial multiclass

In [ ]:
lrp_results_multiclass_misc = {}

for subj in xai_subset:
    print(f"Computing {subj}...")
    misc_out = compute_lrp(
        subject=subj,
        checkpoint_tag='multiclass_16ch',
        n_classes=4,
        use_multiclass=True,
        stopping_criterion='kappa',
        first_layer_rule='epsilon',
        trial_type='misclassified',
    )
    misc_attrs, misc_true, misc_pred, misc_mean = misc_out
    lrp_results_multiclass_misc[subj] = misc_out
    print(f"  {len(misc_attrs)} misclassified trials")
    if len(misc_attrs) > 0:
        # show confusion pattern
        for true_cls in range(4):
            for pred_cls in range(4):
                if true_cls == pred_cls:
                    continue
                count = ((misc_true == true_cls) & (misc_pred == pred_cls)).sum()
                if count > 0:
                    print(f"    true={true_cls} predicted as {pred_cls}: {count}")

print(f"\nDone: {list(lrp_results_multiclass_misc.keys())}")

In [ ]:
class_names = ['left', 'right', 'both-fists', 'both-feet']
unilateral = {0, 1}
bilateral = {2, 3}

print("=== Unilateral -> Bilateral ===")
for subj in xai_subset:
    misc_attrs, misc_true, misc_pred, misc_mean = lrp_results_multiclass_misc[subj]
    mask = (np.isin(misc_true, list(unilateral)) &
            np.isin(misc_pred, list(bilateral)))
    count = mask.sum()
    if count > 0:
        print(f"  {subj}: {count} trials")
        for tc in [0,1]:
            for pc in [2,3]:
                n = ((misc_true==tc) & (misc_pred==pc)).sum()
                if n > 0:
                    print(f"    {class_names[tc]}->{class_names[pc]}: {n}")

print("\n=== Bilateral -> Unilateral ===")
for subj in xai_subset:
    misc_attrs, misc_true, misc_pred, misc_mean = lrp_results_multiclass_misc[subj]
    mask = (np.isin(misc_true, list(bilateral)) &
            np.isin(misc_pred, list(unilateral)))
    count = mask.sum()
    if count > 0:
        print(f"  {subj}: {count} trials")
        for tc in [2,3]:
            for pc in [0,1]:
                n = ((misc_true==tc) & (misc_pred==pc)).sum()
                if n > 0:
                    print(f"    {class_names[tc]}->{class_names[pc]}: {n}")

print("\n=== Cross-Unilateral (left<->right) ===")
for subj in xai_subset:
    misc_attrs, misc_true, misc_pred, misc_mean = lrp_results_multiclass_misc[subj]
    mask = (np.isin(misc_true, list(unilateral)) &
            np.isin(misc_pred, list(unilateral)) &
            (misc_true != misc_pred))
    count = mask.sum()
    if count > 0:
        print(f"  {subj}: {count} trials")
        for tc in [0,1]:
            for pc in [0,1]:
                if tc == pc:
                    continue
                n = ((misc_true==tc) & (misc_pred==pc)).sum()
                if n > 0:
                    print(f"    {class_names[tc]}->{class_names[pc]}: {n}")

In [ ]:
# combined single image
plot_misclassified_combined(lrp_results_multiclass_misc, xai_subjects)

# three separate figures
for etype in ['u2b', 'b2u', 'cu']:
    plot_misclassified_by_type(lrp_results_multiclass_misc,
                                xai_subjects, error_type=etype)

### stopping criteria checks

In [ ]:
# execution
plot_stopping_criterion_consistency(
    lrp_results, lrp_results_composite,
    subjects=consistency_subjects
)

### Save data func_call

In [ ]:
print("Starting binary CDAN-16ch LRP, full 106 subjects...")
binary_completed, binary_skipped, binary_failed = run_lrp_full(
    valid_subjects=valid_subjects,
    checkpoint_tag='cdan_16ch_binary',
    attr_save_path=binary_attr_path,
    n_classes=2,
    use_multiclass=False,
    first_layer_rule='epsilon',
)

In [ ]:
print("Starting multiclass-16ch LRP, full 106 subjects...")
mc_completed, mc_skipped, mc_failed = run_lrp_full(
    valid_subjects=valid_subjects,
    checkpoint_tag='multiclass_16ch',
    attr_save_path=multiclass_attr_path,
    n_classes=4,
    use_multiclass=True,
    first_layer_rule='epsilon',
)

### Grouping on all subjects post testing

In [ ]:
# load both configs
print("Loading binary attributions...")
binary_attrs_106 = load_lrp_attributions(
    binary_attr_path, valid_subjects, n_classes=2
)

print("Loading multiclass attributions...")
mc_attrs_106 = load_lrp_attributions(
    multiclass_attr_path, valid_subjects, n_classes=4
)

In [ ]:
with open(results_path / 'loso_results_multiclass_16ch_with_checkpoints.json') as f:
    mc_results = json.load(f)

# check if per_class_accuracy was saved for a few subjects
for subj in ['S001', 'S037', 'S006', 'S018']:
    entry = mc_results[subj]['kappa_stopping']
    print(f"{subj}: per_class_accuracy = {entry.get('final_per_class_accuracy', 'NOT SAVED')}")

In [ ]:
ckpt = torch.load(
    results_path / "checkpoints" / "multiclass_16ch" / "S001_kappa.pt",
    map_location="cpu", weights_only=False
)
model = DANN(
    n_channels=16, n_timepoints=480, n_classes=4,
    embed_dim=128, dropout_rate=0.25,
    use_conditional_discriminator=True,
    feature_extractor_type='eegnet'
)
model.load_state_dict(ckpt)
model.eval()

source_loader, target_loader, target_eval_loader, y_target = prepare_dataloaders(
    test_subject_id='S001', valid_subjects=valid_subjects,
    use_16ch=True, use_multiclass=True
)

acc, kappa, auc, per_class_acc, conf_mat = evaluate(model, target_eval_loader)
print(f"S001: kappa={kappa:.4f}")
print(f"per_class_acc: {per_class_acc}")

In [ ]:
per_class_path = results_path / "per_class_accuracy"
per_class_path.mkdir(parents=True, exist_ok=True)
print(f"Folder created: {per_class_path}")

In [ ]:
# # delete existing file if it only has kappa
# import os
# existing = per_class_path / "multiclass_16ch_per_class_accuracy.json"
# if existing.exists():
#     os.remove(existing)
#     print("Removed existing single-criterion file, will recompute both.")

# run it
per_class_results = compute_per_class_accuracy_106(
    valid_subjects=valid_subjects,
    checkpoint_tag='multiclass_16ch',
    save_path=per_class_path,
    n_classes=4,
    use_multiclass=True,
    use_16ch=True
)

In [ ]:
# load per-class results
with open(per_class_path / 'multiclass_16ch_per_class_accuracy.json') as f:
    per_class_data = json.load(f)

# peek at one subject to confirm structure
print(json.dumps(per_class_data['S001']['kappa'], indent=2))

In [ ]:
check_subjects = ['S007', 'S037', 'S006', 'S039', 'S020', 'S018']

for subj in check_subjects:
    pca = per_class_data[subj]['kappa']['per_class_accuracy']
    print(f"{subj}:")
    print(f"  left={float(pca['0']):.3f}, right={float(pca['1']):.3f}, "
          f"both-fists={float(pca['2']):.3f}, both-feet={float(pca['3']):.3f}")
    print(f"  right class: {float(pca['1']):.3f}")

In [ ]:
# get per-class accuracy for all 106, kappa stopping
all_per_class = {}
for subj in valid_subjects:
    pca = per_class_data[subj]['kappa']['per_class_accuracy']
    all_per_class[subj] = {
        int(cls): float(acc) for cls, acc in pca.items()
    }

# for each subject, find the lowest class and its accuracy
print("Distribution of minimum class accuracy across 106 subjects:")
min_accs = [min(all_per_class[s].values()) for s in valid_subjects]
print(f"  Mean: {np.mean(min_accs):.3f}")
print(f"  Std: {np.std(min_accs):.3f}")
print(f"  Min: {np.min(min_accs):.3f}")
print(f"  Max: {np.max(min_accs):.3f}")
print(f"  Below 0.1: {sum(1 for x in min_accs if x < 0.1)}")
print(f"  Below 0.2: {sum(1 for x in min_accs if x < 0.2)}")
print(f"  Below 0.3: {sum(1 for x in min_accs if x < 0.3)}")

# spread within subject (max - min), indicator of selective collapse
print("\nDistribution of within-subject class spread (max-min):")
spreads = [max(all_per_class[s].values()) - min(all_per_class[s].values())
           for s in valid_subjects]
print(f"  Mean: {np.mean(spreads):.3f}")
print(f"  Std: {np.std(spreads):.3f}")
print(f"  Above 0.4: {sum(1 for x in spreads if x > 0.4)}")
print(f"  Above 0.5: {sum(1 for x in spreads if x > 0.5)}")
print(f"  Below 0.2: {sum(1 for x in spreads if x < 0.2)}")

In [ ]:
pca_s006 = all_per_class['S006']
spread_s006 = max(pca_s006.values()) - min(pca_s006.values())
min_s006 = min(pca_s006.values())
print(f"S006 spread: {spread_s006:.3f}, min class: {min_s006:.3f}")
print(f"S006 per class: {pca_s006}")
print(f"Which class collapsed: {min(pca_s006, key=pca_s006.get)}")

In [ ]:


group_a, group_b, neither = classify_failure_groups(all_per_class, all_kappas)

print(f"Group A (selective collapse): {len(group_a)} subjects")
for s in sorted(group_a, key=lambda x: x['kappa']):
    print(f"  {s['subject']}: kappa={s['kappa']}, "
          f"collapsed=class {s['collapsed_class']}, "
          f"spread={s['spread']}, min_acc={s['min_acc']}")

print(f"\nGroup B (flat near-chance): {len(group_b)} subjects")
for s in sorted(group_b, key=lambda x: x['kappa']):
    print(f"  {s['subject']}: kappa={s['kappa']}, spread={s['spread']}")

print(f"\nNeither: {len(neither)} subjects")
print(f"Total: {len(group_a) + len(group_b) + len(neither)}")

In [ ]:
check_b = ['S039', 'S097', 'S107']
for subj in check_b:
    pca = all_per_class[subj]
    spread = max(pca.values()) - min(pca.values())
    kappa = all_kappas[subj]
    print(f"{subj}: kappa={kappa:.3f}, spread={spread:.3f}, per_class={pca}")

### Category for all 106 - multiclass

In [ ]:
# build category groups for all 106
group_a_subjects = [s['subject'] for s in group_a]
group_b_subjects = [s['subject'] for s in group_b]

category_groups_106 = {}
for subj in valid_subjects:
    if all_kappas[subj] < 0.1:
        category_groups_106[subj] = 'negative_transfer'
    elif subj in group_a_subjects:
        category_groups_106[subj] = 'group_a_collapse'
    elif subj in group_b_subjects:
        category_groups_106[subj] = 'group_b_flat'
    else:
        category_groups_106[subj] = 'neither'

# quick check
from collections import Counter
print(Counter(category_groups_106.values()))

# confirm our known subjects
for subj in ['S007', 'S037', 'S006', 'S018', 'S020', 'S039']:
    print(f"  {subj}: {category_groups_106[subj]}")

In [ ]:
neg_transfer = [s for s, g in category_groups_106.items()
                if g == 'negative_transfer']
print(f"Negative transfer: {neg_transfer}")
for s in neg_transfer:
    pca = all_per_class[s]
    print(f"  {s}: kappa={all_kappas[s]:.4f}, per_class={pca}")

### plot_group_mean_attribution checks

In [ ]:
# check structure
subj = 'S007'
print(type(binary_attrs_106[subj]))
print(binary_attrs_106[subj].keys())
print(binary_attrs_106[subj]['kappa'].keys())
print(type(binary_attrs_106[subj]['kappa']['mean_by_class']))
print(binary_attrs_106[subj]['kappa']['mean_by_class'].keys())

In [ ]:
plot_group_mean_attribution(
    binary_attrs_106, category_groups_106,
    score_groups, valid_subjects,
    all_kappas
)

### compute_within_group_consistency - call

In [ ]:
# Toggle options:
# config: 'binary' uses binary_attrs_106, n_classes=2
#         'multiclass' uses mc_attrs_106, n_classes=4
# grouping: 'category' uses category_groups_106
#           'score' uses score_groups
# criterion: 'kappa' or 'composite'

class_names_2 = ['left', 'right']
class_names_4 = ['left', 'right', 'both-fists', 'both-feet']

# binary, category grouping
print("=== Binary | Category grouping ===")
res_bin_cat = compute_within_group_consistency(
    attrs_106=binary_attrs_106,
    group_assignments=category_groups_106,
    valid_subjects=valid_subjects,
    config='binary', criterion='kappa', n_classes=2
)
print_consistency_results(res_bin_cat, n_classes=2, class_names=class_names_2)

# binary, score grouping
print("\n=== Binary | Score grouping ===")
res_bin_score = compute_within_group_consistency(
    attrs_106=binary_attrs_106,
    group_assignments=score_groups,
    valid_subjects=valid_subjects,
    config='binary', criterion='kappa', n_classes=2
)
print_consistency_results(res_bin_score, n_classes=2, class_names=class_names_2)

# multiclass, category grouping
print("\n=== Multiclass | Category grouping ===")
res_mc_cat = compute_within_group_consistency(
    attrs_106=mc_attrs_106,
    group_assignments=category_groups_106,
    valid_subjects=valid_subjects,
    config='multiclass', criterion='kappa', n_classes=4
)
print_consistency_results(res_mc_cat, n_classes=4, class_names=class_names_4)

# multiclass, score grouping
print("\n=== Multiclass | Score grouping ===")
res_mc_score = compute_within_group_consistency(
    attrs_106=mc_attrs_106,
    group_assignments=score_groups,
    valid_subjects=valid_subjects,
    config='multiclass', criterion='kappa', n_classes=4
)
print_consistency_results(res_mc_score, n_classes=4, class_names=class_names_4)

### compute_between_group_contrast

In [ ]:
print("=== Between-group contrast | Binary | Category ===")
res_bg_bin_cat, _ = compute_between_group_contrast(
    binary_attrs_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=2
)
print_between_group_results(res_bg_bin_cat, n_classes=2,
                             class_names=class_names_2)

print("\n=== Between-group contrast | Binary | Score ===")
res_bg_bin_score, _ = compute_between_group_contrast(
    binary_attrs_106, score_groups,
    valid_subjects, criterion='kappa', n_classes=2
)
print_between_group_results(res_bg_bin_score, n_classes=2,
                             class_names=class_names_2)

print("\n=== Between-group contrast | Multiclass | Category ===")
res_bg_mc_cat, _ = compute_between_group_contrast(
    mc_attrs_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=4
)
print_between_group_results(res_bg_mc_cat, n_classes=4,
                             class_names=class_names_4)

print("\n=== Between-group contrast | Multiclass | Score ===")
res_bg_mc_score, _ = compute_between_group_contrast(
    mc_attrs_106, score_groups,
    valid_subjects, criterion='kappa', n_classes=4
)
print_between_group_results(res_bg_mc_score, n_classes=4,
                             class_names=class_names_4)

### attribution-performance correlation

In [ ]:
class_names_2 = ['left', 'right']
class_names_4 = ['left', 'right', 'both-fists', 'both-feet']

print("=== Attribution-performance correlation ===")
print("\nBinary | Category | neither group | C3, C4, CP3")
res_corr_bin = compute_attribution_performance_correlation(
    binary_attrs_106, category_groups_106, all_kappas,
    valid_subjects, target_group='neither',
    target_channels=['C3', 'C4', 'CP3'],
    criterion='kappa', n_classes=2
)
for cls, r in res_corr_bin.items():
    if r:
        print(f"  class {class_names_2[cls]}: r={r['r']:.4f} (n={r['n']})")

print("\nMulticlass | Category | neither group | C3, C4, CP3")
res_corr_mc = compute_attribution_performance_correlation(
    mc_attrs_106, category_groups_106, all_kappas,
    valid_subjects, target_group='neither',
    target_channels=['C3', 'C4', 'CP3'],
    criterion='kappa', n_classes=4
)
for cls, r in res_corr_mc.items():
    if r:
        print(f"  class {class_names_4[cls]}: r={r['r']:.4f} (n={r['n']})")

# also check all 16 channels combined vs just motor channels
print("\nBinary | all channels vs motor channels only (left class, neither)")
res_all_ch = compute_attribution_performance_correlation(
    binary_attrs_106, category_groups_106, all_kappas,
    valid_subjects, target_group='neither',
    target_channels=ch16_list,
    criterion='kappa', n_classes=2
)
print(f"  all channels: r={res_all_ch[0]['r']:.4f}")
print(f"  motor only (C3,C4,CP3): r={res_corr_bin[0]['r']:.4f}")

### compute_within_group_performance_drivers

In [ ]:
print("=== Within-group performance drivers | neither group ===")

print("\nBinary:")
res_drivers_bin = compute_within_group_performance_drivers(
    binary_attrs_106, category_groups_106, all_kappas,
    valid_subjects, target_group='neither',
    criterion='kappa', n_classes=2
)
for k, v in res_drivers_bin.items():
    print(f"  {k}: {v}")

print("\nMulticlass:")
res_drivers_mc = compute_within_group_performance_drivers(
    mc_attrs_106, category_groups_106, all_kappas,
    valid_subjects, target_group='neither',
    criterion='kappa', n_classes=4
)
for k, v in res_drivers_mc.items():
    print(f"  {k}: {v}")

### Check refactor LRP 

In [ ]:
# smoke test: load S007, fold, run one LRP attribution
model = load_model_for_xai('S007', 'cdan_16ch_binary', n_classes=2)
wrapper = DANNClassifierWrapper(model)
folded = fold_batchnorm(wrapper)

X, y = load_subject_data_flexible('S007', use_multiclass=False, use_16ch=True)
test_input = torch.FloatTensor(X[:1])

# verify folding still works
verify_folding(wrapper, folded, test_input)

# one attribution
attrs, labels, mean_by_class = compute_lrp(
    subject='S007',
    checkpoint_tag='cdan_16ch_binary',
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    first_layer_rule='epsilon',
    trial_type='correct',
)
print(f"Trials: {len(attrs)}, shape: {attrs[0].shape}")
print(f"Classes: {list(mean_by_class.keys())}")
print("Smoke test passed")